## 1. Environment Setup & Imports

Initializing the research environment. We import standard data manipulation libraries (`pandas`, `numpy`), statistical tools (`scipy.stats` for Spearman rank correlation), and `plotly` for high-quality, interactive financial visualizations. We also configure the default Plotly template to fit a dark-themed quant research style.

In [ ]:
# ==========================================
# CELL 1: SETUP & IMPORTS
# ==========================================
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import statsmodels.api as sm
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
from statsmodels.tsa.stattools import coint, grangercausalitytests
from statsmodels.tsa.vector_ar.vecm import coint_johansen

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    DEFAULT_HOLD_HORIZONS,
    DEFAULT_HIT_RATIO_THRESHOLD,
    DEFAULT_ROLLING_WINDOW,
    DEFAULT_SENSITIVITY_HORIZONS,
    DEFAULT_SENSITIVITY_WINDOWS,
    DEFAULT_SPEARMAN_THRESHOLD,
    SIGNAL_SESSION_END,
    SIGNAL_SESSION_START,
    SIGNAL_TIMEZONE,
)
from src.data_pipeline import convert_index_timezone, load_market_data
from src.indicators import (
    add_correlation_metrics,
    build_volume_analysis_frame,
    fast_rolling_spearman,
    run_adf,
)
from src.optimization import (
    run_robustness_grid_search,
    run_sensitivity_grid_search,
    run_walk_forward_grid_search,
)
from src.simulation import build_signal_frame, evaluate_event_study, summarize_signal_events

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_dark"

print(f"Environment ready. Project root: {PROJECT_ROOT}")


## 2. Data Loading, Cleaning & Timezone Synchronization

This step is critical for cross-asset analysis to avoid look-ahead bias and misalignment:
* **BTCUSDT** trades 24/7, and its data exported from Binance is natively in `UTC`.
* **NQ Futures (MNQ)** trade on the CME. The IBKR data is in the `US/Central` timezone (Chicago) and includes weekend gaps and intraday maintenance breaks.

We localize the NQ data to Chicago time (handling Daylight Saving Time transitions automatically, like the one on March 10, 2024) and convert it to `UTC`. Finally, an `inner join` automatically removes crypto weekend noise, leaving us with a perfectly synchronized dataset containing only regular futures trading hours.

In [ ]:
# ==========================================
# CELL 2: DATA LOADING, CLEANING & SYNC
# ==========================================
ROLLING_WINDOW = DEFAULT_ROLLING_WINDOW

raw_df = load_market_data()
df = add_correlation_metrics(raw_df, window=ROLLING_WINDOW)

print(f"Data synchronized successfully. Total common observations (1m candles): {len(df)}")
display(df.head())


### STATIONARITY TEST (ADF)

In [ ]:
# ==========================================
# CELL 11: STATIONARITY TEST (ADF)
# ==========================================
series_to_test = {
    'BTC Price (Close)': df['btc_close'],
    'NQ Price (Close)': df['nq_close'],
    'BTC Returns': df['btc_ret'],
    'NQ Returns': df['nq_ret'],
}

results = [run_adf(series, name) for name, series in series_to_test.items()]
adf_df = pd.DataFrame(results).set_index('Series')

print('=' * 75)
print('ADF STATIONARITY TEST RESULTS (subsampled to 5,000 obs)')
print('H0: Unit root present (non-stationary). Reject H0 (p < 0.05) => Stationary.')
print('=' * 75)
display(adf_df)


### COINTEGRATION TESTS

In [ ]:
# Use price levels (I(1) series) — drop NaNs
prices = df[['btc_close', 'nq_close']].dropna()

# Subsample: take 5000 contiguous bars (not random — prices need time order)
SAMPLE_SIZE = 5000
prices_sample = prices.iloc[:SAMPLE_SIZE]
print(f'Using {SAMPLE_SIZE} obs ({prices_sample.index[0]} to {prices_sample.index[-1]})')

# ── A) Engle-Granger Test ──────────────────────────────────────────────────
print('=' * 70)
print('A) ENGLE-GRANGER COINTEGRATION TEST')
print('H0: No cointegration. Reject H0 (p < 0.05) => Cointegrated')
print('=' * 70)

for y_col, x_col in [('btc_close', 'nq_close'), ('nq_close', 'btc_close')]:
    eg_stat, eg_p, eg_crit = coint(prices_sample[y_col], prices_sample[x_col])
    verdict = 'COINTEGRATED ✓' if eg_p < 0.05 else 'NOT cointegrated ✗'
    print(f'  Regressing {y_col} ~ {x_col}:')
    print(f'    EG Stat={eg_stat:.4f} | p={eg_p:.4f} | '
          f'Crit 1%={eg_crit[0]:.4f}, 5%={eg_crit[1]:.4f}, 10%={eg_crit[2]:.4f}')
    print(f'    => {verdict}\n')

# ── B) Johansen Test ──────────────────────────────────────────────────────
print('=' * 70)
print('B) JOHANSEN COINTEGRATION TEST')
print('=' * 70)

johansen_result = coint_johansen(prices_sample, det_order=0, k_ar_diff=1)

trace_stats = johansen_result.lr1
max_eigen   = johansen_result.lr2
crit_trace  = johansen_result.cvt
crit_max    = johansen_result.cvm

print(f'{"r":>4} | {"Trace Stat":>12} | {"Crit 95%":>10} | {"Trace Sig":>10} | '
      f'{"MaxEigen Stat":>14} | {"Crit 95%":>10} | {"MaxEigen Sig":>12}')
print('-' * 90)
for i in range(len(trace_stats)):
    t_sig  = 'YES ✓' if trace_stats[i] > crit_trace[i, 1] else 'NO'
    me_sig = 'YES ✓' if max_eigen[i]  > crit_max[i, 1]   else 'NO'
    print(f'{i:>4} | {trace_stats[i]:>12.4f} | {crit_trace[i,1]:>10.4f} | {t_sig:>10} | '
          f'{max_eigen[i]:>14.4f} | {crit_max[i,1]:>10.4f} | {me_sig:>12}')

# ── C) Spread Analysis (use full dataset — OLS is fast) ───────────────────
print('\n' + '=' * 70)
print('C) COINTEGRATION SPREAD (Equilibrium Residual)')
print('=' * 70)

# OLS on full data is fine — it's just matrix multiplication
ols_model  = sm.OLS(prices['btc_close'], sm.add_constant(prices['nq_close'])).fit()
hedge_ratio = ols_model.params['nq_close']
spread = prices['btc_close'] - hedge_ratio * prices['nq_close']

print(f'Hedge Ratio (beta): {hedge_ratio:.6f}')
print(f'Spread | Mean: {spread.mean():.4f} | Std: {spread.std():.4f} | Skew: {spread.skew():.4f}')

sample_spread = spread.loc['2024-03-11':'2024-03-15']
fig = go.Figure()
fig.add_trace(go.Scatter(x=sample_spread.index, y=sample_spread.values,
                         mode='lines', name='Spread', line=dict(color='#F7931A', width=1)))
fig.add_hline(y=spread.mean(), line_dash='dash', line_color='white', annotation_text='Mean')
fig.add_hline(y=spread.mean() + 2*spread.std(), line_dash='dot', line_color='red', annotation_text='+2σ')
fig.add_hline(y=spread.mean() - 2*spread.std(), line_dash='dot', line_color='green', annotation_text='-2σ')
fig.update_layout(title='Cointegration Spread: BTC − β·NQ (Sample Week)', height=400)
fig.show()

### Lead-Lag Analysis

**Does one asset systematically move before the other?**  
A lead-lag relationship means that past returns of Asset A have predictive power over current returns of Asset B. This is distinct from contemporaneous correlation — it implies a **temporal information edge**.

We use two complementary methods:

1. **Cross-Correlation Function (CCF)**: Computes the Pearson correlation between the return series at various time lags `τ`. A peak at `τ < 0` means NQ leads BTC; at `τ > 0`, BTC leads NQ.
2. **Granger Causality Test**: A formal F-test. "X Granger-causes Y" means past values of X significantly improve the prediction of Y beyond Y's own history. Note: this is **predictive** causality, not structural causality.

> A strong lead-lag from NQ→BTC is highly actionable: NQ futures could serve as a real-time leading indicator for BTC entries.

In [ ]:

import plotly.graph_objects as go
from statsmodels.tsa.stattools import grangercausalitytests

returns = df[['btc_ret', 'nq_ret']].dropna()
MAX_LAG = 30  # Maximum number of bars to look back/forward

# ── A) Cross-Correlation Function (CCF) ───────────────────────────────────
print('=' * 70)
print('A) CROSS-CORRELATION FUNCTION (CCF)')
print(f'Lags = -{MAX_LAG} to +{MAX_LAG} bars')
print('Positive lag τ: BTC leads NQ (BTC return at t-τ correlates with NQ at t)')
print('Negative lag τ: NQ leads BTC')
print('=' * 70)

btc = returns['btc_ret'].values
nq  = returns['nq_ret'].values

lags   = range(-MAX_LAG, MAX_LAG + 1)
ccf_vals = []
for lag in lags:
    if lag == 0:
        corr = np.corrcoef(btc, nq)[0, 1]
    elif lag > 0:
        # BTC leads NQ: correlate btc[:-lag] with nq[lag:]
        corr = np.corrcoef(btc[:-lag], nq[lag:])[0, 1]
    else:
        # NQ leads BTC: correlate nq[:lag] with btc[-lag:]
        corr = np.corrcoef(nq[:lag], btc[-lag:])[0, 1]
    ccf_vals.append(corr)

ccf_df = pd.DataFrame({'Lag': list(lags), 'Correlation': ccf_vals})
ccf_df.set_index('Lag', inplace=True)

# Significance bands (95% CI)
n = len(btc)
sig_bound = 1.96 / np.sqrt(n)

best_lag = ccf_df['Correlation'].abs().idxmax()
best_corr = ccf_df.loc[best_lag, 'Correlation']
print(f'Peak CCF: lag={best_lag} bars, correlation={best_corr:.4f}')
if best_lag < 0:
    print(f'  => NQ leads BTC by {abs(best_lag)} minute(s)')
elif best_lag > 0:
    print(f'  => BTC leads NQ by {best_lag} minute(s)')
else:
    print('  => Contemporaneous (no lead-lag at this resolution)')

# Plot CCF
colors = ['#EF553B' if v < 0 else '#00CC96' for v in ccf_vals]
fig = go.Figure()
fig.add_trace(go.Bar(x=list(lags), y=ccf_vals, marker_color=colors, name='CCF'))
fig.add_hline(y=sig_bound,  line_dash='dot', line_color='white',
              annotation_text=f'+95% CI ({sig_bound:.4f})')
fig.add_hline(y=-sig_bound, line_dash='dot', line_color='white',
              annotation_text=f'-95% CI')
fig.add_vline(x=0, line_dash='solid', line_color='gray')
fig.update_layout(
    title='Cross-Correlation Function: BTC vs NQ Returns<br>'
          '<sup>Lag < 0 = NQ leads BTC | Lag > 0 = BTC leads NQ</sup>',
    xaxis_title='Lag (bars = minutes)',
    yaxis_title='Pearson Correlation',
    height=450
)
fig.show()

# ── B) Granger Causality Test ─────────────────────────────────────────────
print('\n' + '=' * 70)
print('B) GRANGER CAUSALITY TEST')
print('H0: X does NOT Granger-cause Y. Reject H0 (p < 0.05) => causality detected.')
print('=' * 70)

MAX_GRANGER_LAG = 10
GRANGER_SAMPLE  = 5000

# Subsample for speed — Granger fits VAR regressions internally
returns_sample = returns.iloc[:GRANGER_SAMPLE]
print(f'(Subsampled to {GRANGER_SAMPLE} obs for Granger tests)')

def granger_summary(data, cause, effect, max_lag):
    print(f'\n  [{cause} Granger-causes {effect}]')
    test_data = data[[effect, cause]].dropna()
    results = grangercausalitytests(test_data, maxlag=max_lag, verbose=False)
    rows = []
    for lag, res in results.items():
        f_stat = res[0]['ssr_ftest'][0]
        p_val  = res[0]['ssr_ftest'][1]
        rows.append({'Lag': lag, 'F-Stat': round(f_stat, 4),
                     'p-value': round(p_val, 4),
                     'Significant?': 'YES ✓' if p_val < 0.05 else 'NO'})
    df_res = pd.DataFrame(rows).set_index('Lag')
    display(df_res)
    sig_pct = (df_res['Significant?'] == 'YES ✓').mean() * 100
    print(f'  Significant at {sig_pct:.0f}% of tested lags')
    return df_res

gc_nq_btc = granger_summary(returns_sample, 'nq_ret', 'btc_ret', MAX_GRANGER_LAG)
gc_btc_nq = granger_summary(returns_sample, 'btc_ret', 'nq_ret', MAX_GRANGER_LAG)

#### Conclusion for Lead-Lag Analysis:
There is no exploitable lead-lag relationship between BTC and NQ at 1-minute resolution. The correlation is contemporaneous — they react to the same macro information at the same time, not sequentially. This actually strengthens our existing approach (using real-time rolling correlation as the signal) rather than a lagged one.

### Volume-Weighted Analysis

Standard correlation treats every bar equally, regardless of whether it had 100 contracts or 10,000. **High-volume bars carry institutional intent** — they reflect real commitment, not noise.

This section weights the correlation metrics by volume to separate signal from noise:

1. **Volume-Weighted Correlation**: Each bar's contribution to the rolling correlation is proportional to its normalized volume. High-volume periods carry more weight.
2. **VWAP (Volume Weighted Average Price)**: The fair-value benchmark for intraday institutional execution. We compute rolling VWAP for both assets.
3. **Volume Regime Analysis**: We split bars into High/Low volume quintiles and compare the correlation metrics across each quintile.
4. **Combined Volume Score**: A single composite metric that is the product of correlation strength × volume rank — a concise filter for high-confidence, high-conviction alignment.

In [ ]:
# ==========================================
# CELL 15: VOLUME-WEIGHTED ANALYSIS
# ==========================================
vdf = build_volume_analysis_frame(df, ROLLING_WINDOW)

quintile_stats = vdf.groupby('vol_quintile', observed=True)[['corr_spearman', 'vw_corr', 'hit_ratio']].agg(
    ['mean', 'std', 'count']
).round(4)

print("\n" + '=' * 70)
print('VOLUME QUINTILE ANALYSIS')
print('How do correlation metrics differ across volume regimes?')
print('=' * 70)
print(quintile_stats.to_string())

START_DATE = '2024-03-11'
END_DATE = '2024-03-15'
plot_vdf = vdf.loc[START_DATE:END_DATE].copy()

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    row_heights=[0.3, 0.25, 0.25, 0.2],
    subplot_titles=(
        'BTC & NQ vs Rolling VWAP',
        f'Volume-Weighted Corr vs Standard Spearman ({ROLLING_WINDOW}-bar)',
        'Combined Volume (log scale)',
        'Volume-Conviction Score'
    )
)

btc_norm = plot_vdf['btc_close'] / plot_vdf['btc_close'].iloc[0] * 100
nq_norm = plot_vdf['nq_close'] / plot_vdf['nq_close'].iloc[0] * 100
btc_vwap_norm = plot_vdf['btc_vwap'] / plot_vdf['btc_close'].iloc[0] * 100
nq_vwap_norm = plot_vdf['nq_vwap'] / plot_vdf['nq_close'].iloc[0] * 100

fig.add_trace(go.Scatter(x=plot_vdf.index, y=btc_norm, mode='lines',
                         name='BTC', line=dict(color='#F7931A', width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_vdf.index, y=btc_vwap_norm, mode='lines',
                         name='BTC VWAP', line=dict(color='#F7931A', width=1, dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_vdf.index, y=nq_norm, mode='lines',
                         name='NQ', line=dict(color='#00A6DF', width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_vdf.index, y=nq_vwap_norm, mode='lines',
                         name='NQ VWAP', line=dict(color='#00A6DF', width=1, dash='dot')), row=1, col=1)

fig.add_trace(go.Scatter(x=plot_vdf.index, y=plot_vdf['vw_corr'], mode='lines',
                         name='Vol-Weighted Corr', line=dict(color='#FF6B35', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=plot_vdf.index, y=plot_vdf['corr_spearman'], mode='lines',
                         name='Spearman', line=dict(color='#00CC96', width=1, dash='dash')), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

fig.add_trace(go.Scatter(x=plot_vdf.index, y=np.log1p(plot_vdf['combined_vol']), mode='lines',
                         name='Log Combined Vol', line=dict(color='#AB63FA', width=1)), row=3, col=1)

fig.add_trace(go.Scatter(x=plot_vdf.index, y=plot_vdf['conviction_score'], mode='lines',
                         name='Conviction Score', line=dict(color='#FFA15A', width=1)), row=4, col=1)

fig.update_layout(
    title=f'Volume-Weighted Analysis ({START_DATE} to {END_DATE})',
    height=1000, hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1)
)
fig.update_yaxes(title_text='Normalized Price', row=1, col=1)
fig.update_yaxes(title_text='Correlation', range=[-1.1, 1.1], row=2, col=1)
fig.update_yaxes(title_text='Log Volume', row=3, col=1)
fig.update_yaxes(title_text='Score [0-1]', row=4, col=1)
fig.show()

print("\n" + '=' * 70)
print('VOLUME-WEIGHTED METRICS SUMMARY')
print('=' * 70)
summary = vdf[['corr_spearman', 'vw_corr', 'conviction_score']].describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
).T.round(4)
print(summary.to_string())


#### Conclusion for Volume Weighted Analysis:
Volume-weighted analysis reveals that correlation is meaningfully stronger during high-volume periods (vw_corr mean 0.28 vs Spearman 0.25, with the gap widening significantly at the 75th percentile), confirming that institutional flow bars carry disproportionate signal quality. True high-conviction alignment — defined by the composite conviction score above 0.50 — occurs in only the top 5% of all bars, meaning the strategy must be highly selective to avoid trading noise, and the existing Hit Ratio + Spearman gate from the regime engine is well-justified by the data.

## 3. Correlation Metrics Engine (The Math)

In this section, we calculate three distinct metrics over a rolling window to identify true structural alignment between BTC and NQ. 

1. **Rolling Pearson Correlation**: The standard baseline. It measures linear relationships but is highly sensitive to outliers (e.g., a single volatile minute can artificially pump the correlation near 1.0).
2. **Rolling Spearman Rank Correlation**: A non-parametric measure. By evaluating the *rank* of returns rather than their absolute magnitude, it ignores micro-volatility spikes. Since `pandas.rolling().corr()` doesn't support Spearman directly, we use a highly optimized, vectorized NumPy algorithm (`sliding_window_view`) to compute it instantly over millions of rows.
3. **Directional Hit Ratio (Concordance)**: The most intuitive metric. It calculates the percentage of minutes within the rolling window where both assets moved in the exact same direction (both up, both down, or both flat). This directly answers the question: *"Are these markets actually stepping together?"*

In [ ]:
# ==========================================
# CELL 3: ROLLING CORRELATIONS & HIT RATIO
# ==========================================
print(f"Metrics calculated successfully using a {ROLLING_WINDOW}-bar window from src.indicators.")
display(df[['btc_ret', 'nq_ret', 'corr_pearson', 'corr_spearman', 'hit_ratio']].tail(10))


## 4. Global Time Series Visualization (EDA)

Interactive visualization is crucial for understanding the dynamic relationship between assets. We use `plotly` to stack the price action directly above our computed correlation metrics. 

**Note on Memory Management:** Since 1-minute data over 2 years produces hundreds of thousands of rows, rendering the entire dataset interactively will crash the browser. We slice the data into a specific time window (e.g., a specific trading week) to observe the micro-structure and evaluate how Pearson, Spearman, and Hit Ratio react to real price movements.

* **Top Panel**: Normalized Prices (Base 100). Allows direct visual comparison of relative momentum.
* **Middle Panel**: Rolling Pearson vs. Rolling Spearman. Notice how Spearman is generally smoother and less reactive to single-bar volatility spikes.
* **Bottom Panel**: Directional Hit Ratio. Shows the percentage of aligned bars. A value of 0.5 implies random walk (no directional correlation), while values > 0.7 or < 0.3 indicate strong regime alignment or divergence.

In [ ]:
# ==========================================
# CELL 4: INTERACTIVE VISUALIZATION
# ==========================================

# 1. Define the slice for visualization to prevent browser crashes
# Let's look at the first full trading week in the dataset
START_DATE = '2024-03-11'
END_DATE = '2024-03-15'

plot_df = df.loc[START_DATE:END_DATE].copy()

# 2. Normalize prices to Base 100 for proper visual scale comparison
plot_df['btc_norm'] = (plot_df['btc_close'] / plot_df['btc_close'].iloc[0]) * 100
plot_df['nq_norm'] = (plot_df['nq_close'] / plot_df['nq_close'].iloc[0]) * 100

# 3. Create a multi-panel figure
fig = make_subplots(
    rows=3, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.5, 0.25, 0.25],
    subplot_titles=(
        "Normalized Prices (Base 100): BTC vs NQ", 
        f"Rolling {ROLLING_WINDOW}-bar Correlations", 
        f"Directional Hit Ratio ({ROLLING_WINDOW}-bar)"
    )
)

# --- ROW 1: Normalized Prices ---
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['btc_norm'], 
                         mode='lines', name='BTC (Base 100)', line=dict(color='#F7931A', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['nq_norm'], 
                         mode='lines', name='NQ (Base 100)', line=dict(color='#00A6DF', width=1.5)), row=1, col=1)

# --- ROW 2: Pearson vs Spearman ---
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['corr_pearson'], 
                         mode='lines', name='Pearson', line=dict(color='#EF553B', width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['corr_spearman'], 
                         mode='lines', name='Spearman', line=dict(color='#00CC96', width=1.5)), row=2, col=1)

# Add a zero-line reference for correlations
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)

# --- ROW 3: Hit Ratio ---
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['hit_ratio'], 
                         mode='lines', name='Hit Ratio', line=dict(color='#AB63FA', width=1)), row=3, col=1)

# Add reference lines for Hit Ratio (0.5 = random, 0.7 = strong directional agreement)
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=3, col=1)
fig.add_hline(y=0.7, line_dash="dot", line_color="rgba(0, 204, 150, 0.5)", row=3, col=1)

# 4. Layout formatting
fig.update_layout(
    title=f"Cross-Asset Alignment Analysis ({START_DATE} to {END_DATE})",
    height=900,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Normalized Value", row=1, col=1)
fig.update_yaxes(title_text="Correlation (-1 to 1)", range=[-1.1, 1.1], row=2, col=1)
fig.update_yaxes(title_text="Hit Ratio (0 to 1)", range=[-0.05, 1.05], row=3, col=1)

fig.show()

## 5. Statistical Summary of Metrics

Visualizing the time series gives us intuition, but quantitative research requires hard statistics. In this cell, we generate a statistical summary table of our correlation metrics across the entire 2-year dataset.

We specifically look at:
* **Mean & Median (50%)**: To establish the baseline "random walk" state.
* **Tails (5%, 95%)**: To understand the extreme thresholds of the metrics.
* **Regime Frequencies**: A quick calculation to see exactly what percentage of the time the markets are in a state of high directional agreement (Hit Ratio $\ge$ 0.70 and $\ge$ 0.80).

In [ ]:
# ==========================================
# CELL 5: STATISTICAL SUMMARY TABLE
# ==========================================

# 1. Isolate the metrics for statistical analysis
metrics_df = df[['corr_pearson', 'corr_spearman', 'hit_ratio']]

# 2. Generate summary statistics with custom distribution percentiles
summary_table = metrics_df.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

# 3. Display the summary table cleanly
print("="*60)
print("GLOBAL STATISTICAL SUMMARY (ENTIRE DATASET)")
print("="*60)
display(summary_table.round(4))

# 4. Calculate specific "High Agreement" frequencies
# How often are BTC and NQ moving in the exact same direction > 70% or > 80% of the time?
hr_gt_70 = (df['hit_ratio'] >= 0.70).mean() * 100
hr_gt_80 = (df['hit_ratio'] >= 0.80).mean() * 100
hr_lt_30 = (df['hit_ratio'] <= 0.30).mean() * 100

print("\n" + "="*60)
print("EXTREME REGIME FREQUENCIES")
print("="*60)
print(f"Time in High Positive Agreement (Hit Ratio >= 70%) : {hr_gt_70:>5.2f}%")
print(f"Time in Extreme Positive Agreement (Hit Ratio >= 80%): {hr_gt_80:>5.2f}%")
print(f"Time in High Negative Divergence (Hit Ratio <= 30%)  : {hr_lt_30:>5.2f}%")

# Base Random Walk Check
median_hr = summary_table.loc['hit_ratio', '50%']
print(f"\nBaseline Market State (Median Hit Ratio): {median_hr:.2f}")
print("*(A value near 0.50 confirms that most of the time, the micro-movements are just uncorrelated noise)*")

## 6. Intraday Seasonality Analysis (Time of Day)

A common pitfall in cross-asset correlation strategies is falling for "fake" signals generated during low-liquidity periods (e.g., Asian session or early US pre-market). During these times, a single erratic print can artificially pump the standard deviation and rolling correlation.

To filter out this noise, we analyze the average metrics based on the **Time of Day**. 
We temporarily convert the index to New York time (`America/New_York`) to intuitively observe the behavioral shift when Regular Trading Hours (RTH) begin at 09:30 AM EST.

In [ ]:
# ==========================================
# CELL 6: INTRADAY SEASONALITY (TIME OF DAY)
# ==========================================
df_ny = convert_index_timezone(df, SIGNAL_TIMEZONE)
df_ny['hour'] = df_ny.index.hour

hourly_stats = df_ny.groupby('hour')[['hit_ratio', 'corr_spearman', 'corr_pearson']].mean()

df_ny['extreme_regime'] = (df_ny['hit_ratio'] >= 0.80).astype(int)
hourly_extreme_freq = df_ny.groupby('hour')['extreme_regime'].sum()
hourly_extreme_pct = (hourly_extreme_freq / hourly_extreme_freq.sum()) * 100

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        'Average Correlation & Hit Ratio by Hour of Day (NY Time)',
        'Distribution of Extreme Agreement Signals (Hit Ratio >= 0.80) by Hour %'
    )
)

fig.add_trace(go.Bar(x=hourly_stats.index, y=hourly_stats['hit_ratio'],
                     name='Avg Hit Ratio', marker_color='#AB63FA'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_stats.index, y=hourly_stats['corr_spearman'],
                         mode='lines+markers', name='Avg Spearman', line=dict(color='#00CC96', width=3)), row=1, col=1)
fig.add_vline(x=9.5, line_width=2, line_dash='dash', line_color='white',
              annotation_text='US Market Open (09:30)', annotation_position='top right', row=1, col=1)

fig.add_trace(go.Bar(x=hourly_extreme_pct.index, y=hourly_extreme_pct.values,
                     name='% of Total Extreme Signals', marker_color='#EF553B'), row=2, col=1)
fig.add_vline(x=9.5, line_width=2, line_dash='dash', line_color='white', row=2, col=1)

fig.update_layout(
    height=800,
    hovermode='x unified',
    xaxis2=dict(title='Hour of the Day (New York Time)', tickmode='linear', tick0=0, dtick=1),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.update_yaxes(title_text='Metric Value', row=1, col=1)
fig.update_yaxes(title_text='% of Signals', row=2, col=1)

fig.show()


## 6.5 Intraday Seasonality Data Export

To quantitatively validate the visual heatmap from the previous step, we compile the hourly average metrics and the distribution of extreme signals into a single DataFrame. This table is then displayed and exported to a CSV file for further analysis and hardcoding trading session filters.

In [ ]:
# ==========================================
# CELL 6.5: EXPORT INTRADAY SEASONALITY DATA
# ==========================================

# Combine the data from Cell 6 into a single summary table
seasonality_table = hourly_stats.copy()
seasonality_table['extreme_signals_pct'] = hourly_extreme_pct

# Round for readability
seasonality_table = seasonality_table.round(4)

# Rename columns for clarity
seasonality_table.columns = [
    'Avg Hit Ratio', 
    'Avg Spearman', 
    'Avg Pearson', 
    '% of Total Extreme Signals'
]

print("="*70)
print("INTRADAY SEASONALITY SUMMARY (NEW YORK TIME)")
print("="*70)
display(seasonality_table)

## 7. Signal Quality Matrix (Spearman vs. Hit Ratio)

To avoid falling into the trap of "mathematically scrunched" correlations during low-liquidity hours, we must evaluate the relationship between continuous correlation (`Spearman`) and discrete bar-by-bar alignment (`Hit Ratio`).

Here we plot a **2D Density Heatmap**. 
* **Toxic Zones**: Areas with high correlation (e.g., Spearman > 0.7) but low Hit Ratio (~0.5). These represent illiquid, erratic movements where math breaks down.
* **Alpha Zones**: Areas where both Spearman is high AND Hit Ratio is high (> 0.7). This is true, structural institutional flow.

In [ ]:
# ==========================================
# CELL 7: SIGNAL QUALITY 2D MATRIX
# ==========================================
import plotly.express as px

# We use the New York time dataframe (df_ny)
# To keep the plot extremely fast and avoid overplotting, we randomly sample 10% of the data
# (Statistically, the distribution will remain exactly the same)
sample_df = df_ny.sample(frac=0.10, random_state=42)

# Create a 2D Density Heatmap
fig = px.density_heatmap(
    sample_df, 
    x="corr_spearman", 
    y="hit_ratio", 
    nbinsx=50, 
    nbinsy=20,
    color_continuous_scale="Viridis",
    title="Signal Quality Matrix: Spearman Correlation vs. Directional Hit Ratio"
)

# Add highlight boxes/lines to denote the "Alpha Zone" vs "Toxic Zone"
# Alpha Zone: Spearman > 0.6 AND Hit Ratio >= 0.7
fig.add_shape(type="rect",
    x0=0.6, y0=0.7, x1=1.0, y1=1.0,
    line=dict(color="LimeGreen", width=2, dash="dash"),
    fillcolor="rgba(50, 205, 50, 0.1)"
)
fig.add_annotation(x=0.8, y=0.85, text="ALPHA ZONE<br>(True Flow)", showarrow=False, font=dict(color="LimeGreen"))

# Toxic Zone: Spearman > 0.6 BUT Hit Ratio <= 0.6 (Random walk territory)
fig.add_shape(type="rect",
    x0=0.6, y0=0.3, x1=1.0, y1=0.6,
    line=dict(color="Red", width=2, dash="dash"),
    fillcolor="rgba(255, 0, 0, 0.1)"
)
fig.add_annotation(x=0.8, y=0.45, text="TOXIC ZONE<br>(Fake Pump / Noise)", showarrow=False, font=dict(color="Red"))


fig.update_layout(
    height=700,
    xaxis_title="Rolling Spearman Rank Correlation",
    yaxis_title="Directional Hit Ratio (Concordance)",
    coloraxis_colorbar=dict(title="Density (Observations)")
)

fig.show()

## 7.5 Signal Quality Matrix Data Export

Visual heatmaps are great for intuition, but we need hard percentages to define algorithmic thresholds. This cell creates a cross-tabulation (2D frequency matrix) of our `Hit Ratio` and `Spearman Correlation`. 

It divides the metrics into distinct bins and calculates the percentage of total market time spent in each combination. Most importantly, it calculates the exact weight of our **Alpha Zone** (true directional flow) versus the **Toxic Zone** (mathematical correlation illusion). The matrix is exported to CSV.

In [ ]:
# ==========================================
# CELL 7.5: SIGNAL QUALITY MATRIX EXPORT
# ==========================================
import numpy as np

# 1. Define bins for Hit Ratio and Spearman Correlation
hit_ratio_bins = np.arange(0.0, 1.1, 0.1)  # [0.0, 0.1, 0.2 ... 1.0]
spearman_bins = np.arange(-1.0, 1.2, 0.2)  # [-1.0, -0.8, -0.6 ... 1.0]

# 2. Categorize the data into these bins
# We use df_ny (our full dataset with New York timezone)
df_ny['hr_bin'] = pd.cut(df_ny['hit_ratio'], bins=hit_ratio_bins, include_lowest=True)
df_ny['spearman_bin'] = pd.cut(df_ny['corr_spearman'], bins=spearman_bins, include_lowest=True)

# 3. Create a Cross-Tabulation (Frequency Matrix)
# normalize='all' gives us the percentage of total observations, multiplied by 100 for readability
signal_matrix = pd.crosstab(df_ny['hr_bin'], df_ny['spearman_bin'], normalize='all') * 100
signal_matrix = signal_matrix.round(2) # Round to 2 decimal places

# 4. Display the Matrix
print("="*85)
print("SIGNAL QUALITY MATRIX: % OF TOTAL OBSERVATIONS")
print("Rows: Hit Ratio | Columns: Spearman Correlation")
print("="*85)
display(signal_matrix)

# 5. Calculate Specific Zone Weights
# Alpha Zone: True structural flow
alpha_zone_mask = (df_ny['corr_spearman'] >= 0.6) & (df_ny['hit_ratio'] >= 0.7)
alpha_zone_pct = (alpha_zone_mask.sum() / len(df_ny)) * 100

# Toxic Zone: High math correlation, but poor bar-by-bar directional match
toxic_zone_mask = (df_ny['corr_spearman'] >= 0.6) & (df_ny['hit_ratio'] <= 0.6)
toxic_zone_pct = (toxic_zone_mask.sum() / len(df_ny)) * 100

print("\n" + "="*85)
print("ZONE ANALYSIS (KEY THRESHOLDS)")
print("="*85)
print(f"[ALPHA ZONE] High Spearman (>=0.6) & High Hit Ratio (>=0.7): {alpha_zone_pct:.2f}% of all market time")
print(f"[TOXIC ZONE] High Spearman (>=0.6) & Low Hit Ratio (<=0.6) : {toxic_zone_pct:.2f}% of all market time")

## 8. Signal Generation & Event Clustering

Using the insights from our EDA, we now define the hard rules for entering a trade. 
A valid **Regime Signal** occurs when:
1. **Time Filter**: Between 09:30 and 16:00 NY Time (Regular Trading Hours).
2. **Directional Flow**: Hit Ratio >= 0.70.
3. **Mathematical Validation**: Spearman Correlation >= 0.60.

Because these conditions will persist over multiple consecutive 1-minute bars, we group adjacent signal bars into **Signal Clusters (Events)**. This tells us exactly how many distinct trading opportunities occurred over the 2-year period.

In [ ]:
# ==========================================
# CELL 8: SIGNAL LOGIC & CLUSTERING
# ==========================================
signal_df = build_signal_frame(
    df_ny,
    session_start=SIGNAL_SESSION_START,
    session_end=SIGNAL_SESSION_END,
    hit_ratio_threshold=DEFAULT_HIT_RATIO_THRESHOLD,
    spearman_threshold=DEFAULT_SPEARMAN_THRESHOLD,
)
signal_summary = summarize_signal_events(signal_df)

print('=' * 60)
print('REGIME SIGNAL ANALYSIS')
print('=' * 60)
print(f"Total evaluated 1-min bars: {signal_summary['bars']:,}")
print(f"Total minutes meeting ALL conditions: {signal_summary['signal_minutes']:,}")
print(f"Number of distinct Trading Events (Clusters): {signal_summary['total_events']:,}")
print(f"Total Trading Days in Dataset: {signal_summary['trading_days']:,}")
print(f"Average Actionable Events per Day: {signal_summary['events_per_day']:.2f}")

print("\nFirst 5 Signal Event Timestamps:")
display(signal_df[signal_df['event_start']][['hit_ratio', 'corr_spearman']].head(5))


## 9. Forward Returns Analysis (Event Study)

Identifying a statistical regime is only half the battle. The ultimate question for a Quant is: *"Does the price actually follow through after the signal is triggered?"*

In this final cell, we conduct a **Forward Returns Analysis**. We:
1. Determine the prevailing direction of the regime (Up or Down) using the 15-minute momentum at the time of the event.
2. Look ahead exactly 5, 10, and 15 minutes after the `event_start` trigger.
3. Calculate the directional returns. A **positive percentage** indicates trend continuation (profit), while a negative percentage indicates a reversal (loss).

In [ ]:
# ==========================================
# CELL 9: FORWARD RETURNS (EVENT STUDY)
# ==========================================
horizons = list(DEFAULT_HOLD_HORIZONS)
events_df, forward_summary, win_rates = evaluate_event_study(
    signal_df,
    horizons=horizons,
    momentum_window=ROLLING_WINDOW,
)

print('=' * 80)
print('AVERAGE DIRECTIONAL FORWARD RETURNS (IN BASIS POINTS)')
print('Note: Positive mean/median means the trend successfully continued after the signal.')
print('10 Basis Points (bps) = 0.1%')
print('=' * 80)
display(forward_summary.round(2))

print("\n" + '=' * 80)
print('TIME-BASED WIN RATES (% of signals with positive follow-through)')
print('=' * 80)
for row in win_rates.itertuples(index=False):
    print(
        f'Holding for {row.horizon_minutes:>2} mins | '
        f'NQ Win Rate: {row.nq_win_rate_pct:.1f}% | '
        f'BTC Win Rate: {row.btc_win_rate_pct:.1f}%'
    )


## 10. Multi-Dimensional Robustness Analysis (Expanded Grid Search)

The original sensitivity check only varied **window size** and **holding horizon** while keeping the regime thresholds fixed. That is useful for intuition, but it is not enough to claim the signal is robust.

In this upgraded section, we run **three layers of validation**:
1. A baseline **Window x Hold** sweep to see how persistence changes across lookback and holding periods.
2. A broader **Window x Hold x Hit-Ratio Threshold x Spearman Threshold** search with a chronological **70/30 in-sample vs out-of-sample split**.
3. A stricter **monthly walk-forward validation** where each parameter set is tested across repeated out-of-sample months using only prior months as the training history.

This third layer is the closest thing in the notebook to a real deployment test, because it shows whether the same configuration keeps working across changing market regimes instead of only surviving one lucky split.


In [ ]:
# ==========================================
# CELL 10: ROBUST SENSITIVITY GRID SEARCH
# ==========================================
test_windows         = list(np.arange(5, 65, 5))    # [5, 10, 15, ..., 60] — 12 values
hold_horizons        = list(np.arange(5, 65, 5))    # [5, 10, 15, ..., 60] — 12 values
hit_ratio_thresholds = np.arange(0.5, 1.01, 0.05)  # [0.50, 0.55, ..., 1.00] — 11 values
spearman_thresholds  = np.arange(0.5, 1.1, 0.1)    # [0.50, 0.60, ..., 1.00] — 6 values
min_events_per_side  = 30

print('=' * 80)
print('GRID SEARCH PARAMETERS')
print('=' * 80)
print(f'  Rolling Windows       : {test_windows[0]}–{test_windows[-1]} bars, step=5  → {test_windows}')
print(f'  Hold Horizons         : {hold_horizons[0]}–{hold_horizons[-1]} mins, step=5  → {hold_horizons}')
print(f'  Hit Ratio Thresholds  : {hit_ratio_thresholds[0]:.2f}–{hit_ratio_thresholds[-1]:.2f}, step=0.05  ({len(hit_ratio_thresholds)} values)')
print(f'  Spearman Thresholds   : {spearman_thresholds[0]:.2f}–{spearman_thresholds[-1]:.2f}, step=0.10  ({len(spearman_thresholds)} values)')
print(f'  Min Events/Side       : {min_events_per_side}')
print(f'  Baseline grid size    : {len(test_windows)} windows × {len(hold_horizons)} horizons = {len(test_windows)*len(hold_horizons):,} cells')
print(f'  Full robustness grid  : {len(test_windows)} × {len(hold_horizons)} × {len(hit_ratio_thresholds)} × {len(spearman_thresholds)} = {len(test_windows)*len(hold_horizons)*len(hit_ratio_thresholds)*len(spearman_thresholds):,} combos')
print('=' * 80)

# ── BASELINE: Window x Hold sweep (thresholds fixed at defaults) ──────────
print('\nRunning baseline Window x Hold sensitivity grid...')
print(f'  (Hit Ratio fixed = {DEFAULT_HIT_RATIO_THRESHOLD}, Spearman fixed = {DEFAULT_SPEARMAN_THRESHOLD})')
df_wr, df_bps, df_med_bps = run_sensitivity_grid_search(
    df_ny,
    test_windows=test_windows,
    hold_horizons=hold_horizons,
    hit_ratio_threshold=DEFAULT_HIT_RATIO_THRESHOLD,
    spearman_threshold=DEFAULT_SPEARMAN_THRESHOLD,
    session_start=SIGNAL_SESSION_START,
    session_end=SIGNAL_SESSION_END,
)

print("\n" + '=' * 80)
print('WIN RATE MATRIX (%)')
print(f'  What it shows : % of signals with positive follow-through')
print(f'  Rows          : Rolling window size (bars)')
print(f'  Columns       : Hold horizon (minutes)')
print(f'  Fixed params  : Hit Ratio threshold = {DEFAULT_HIT_RATIO_THRESHOLD}, Spearman threshold = {DEFAULT_SPEARMAN_THRESHOLD}')
print('=' * 80)
print(df_wr.round(2).to_string())

print("\n" + '=' * 80)
print('AVERAGE FORWARD RETURNS MATRIX (Basis Points)')
print(f'  What it shows : Mean directional return after signal trigger')
print(f'  Rows          : Rolling window size (bars)')
print(f'  Columns       : Hold horizon (minutes)')
print(f'  Fixed params  : Hit Ratio threshold = {DEFAULT_HIT_RATIO_THRESHOLD}, Spearman threshold = {DEFAULT_SPEARMAN_THRESHOLD}')
print('=' * 80)
print(df_bps.round(2).to_string())

print("\n" + '=' * 80)
print('MEDIAN FORWARD RETURNS MATRIX (Basis Points)')
print(f'  What it shows : Median directional return after signal trigger (outlier-robust)')
print(f'  Rows          : Rolling window size (bars)')
print(f'  Columns       : Hold horizon (minutes)')
print(f'  Fixed params  : Hit Ratio threshold = {DEFAULT_HIT_RATIO_THRESHOLD}, Spearman threshold = {DEFAULT_SPEARMAN_THRESHOLD}')
print('=' * 80)
print(df_med_bps.round(2).to_string())

# ── EXPANDED: Robustness search with 70/30 OOS split ─────────────────────
print(f'\nRunning expanded robustness search (70/30 IS/OOS split)...')
print(f'  Sweeping all {len(test_windows)*len(hold_horizons)*len(hit_ratio_thresholds)*len(spearman_thresholds):,} combos')
print(f'  IS split = first 70% of data, OOS = remaining 30%')
print(f'  Min events required each side = {min_events_per_side}')
robust_results = run_robustness_grid_search(
    df_ny,
    test_windows=test_windows,
    hold_horizons=hold_horizons,
    hit_ratio_thresholds=hit_ratio_thresholds,
    spearman_thresholds=spearman_thresholds,
    session_start=SIGNAL_SESSION_START,
    session_end=SIGNAL_SESSION_END,
    split_ratio=0.70,
    min_events_per_side=min_events_per_side,
)

credible_results = robust_results[robust_results['passes_min_events']].copy()
analysis_results = credible_results.copy()
if analysis_results.empty:
    print(f'No configurations passed the min-event filter of {min_events_per_side}. Falling back to all results.')
    analysis_results = robust_results.copy()

print("\n" + '=' * 80)
print('ROBUSTNESS SEARCH SUMMARY')
print('=' * 80)
print(f'  Total combos evaluated                          : {len(robust_results):,}')
print(f'  Combos passing min-event filter (>={min_events_per_side} each side) : {len(credible_results):,}')
if not analysis_results.empty:
    print(f'  Best OOS avg bps in credible set                : {analysis_results["oos_avg_bps"].max():.2f}')
    print(f'  Best robust score in credible set               : {analysis_results["robust_score"].max():.2f}')

top_cols = [
    'window_bars', 'hold_minutes', 'hit_ratio_threshold', 'spearman_threshold',
    'train_events', 'test_events',
    'is_win_rate_pct', 'oos_win_rate_pct',
    'is_avg_bps', 'oos_avg_bps',
    'win_rate_gap', 'bps_gap', 'robust_score'
]

if not analysis_results.empty:
    print("\nTop 15 robust configurations (ranked by OOS-aware stability score):")
    display(analysis_results[top_cols].head(15).round(2))

    best_by_window_hold = (
        analysis_results
        .sort_values(['robust_score', 'oos_avg_bps', 'oos_win_rate_pct'], ascending=False)
        .groupby(['window_bars', 'hold_minutes'], as_index=False)
        .first()
    )

    pivot_oos_bps      = best_by_window_hold.pivot(index='window_bars', columns='hold_minutes', values='oos_avg_bps')
    pivot_robust_score = best_by_window_hold.pivot(index='window_bars', columns='hold_minutes', values='robust_score')
    pivot_best_hr      = best_by_window_hold.pivot(index='window_bars', columns='hold_minutes', values='hit_ratio_threshold')
    pivot_best_sp      = best_by_window_hold.pivot(index='window_bars', columns='hold_minutes', values='spearman_threshold')

    print("\n" + '=' * 80)
    print('BEST OOS AVG BPS BY WINDOW x HOLD (after threshold optimization + min-event filter)')
    print(f'  What it shows : Best OOS avg return (bps) found across all threshold combos for each Window/Hold pair')
    print(f'  Rows          : Rolling window size (bars) — tested: {test_windows}')
    print(f'  Columns       : Hold horizon (minutes)     — tested: {hold_horizons}')
    print(f'  Selection     : Per cell, the threshold combo with highest robust_score was chosen (see companion tables below)')
    print(f'  IS/OOS split  : 70% in-sample / 30% out-of-sample, chronological')
    print(f'  Min events    : >= {min_events_per_side} signal events required on each side')
    print('=' * 80)
    print(pivot_oos_bps.round(2).to_string())

    print("\n" + '=' * 80)
    print('BEST HIT RATIO THRESHOLD SELECTED PER WINDOW x HOLD CELL')
    print(f'  What it shows : Which Hit Ratio threshold produced the best robust_score for each cell above')
    print(f'  Rows          : Rolling window size (bars) — tested: {test_windows}')
    print(f'  Columns       : Hold horizon (minutes)     — tested: {hold_horizons}')
    print(f'  Threshold range swept : {hit_ratio_thresholds[0]:.2f} – {hit_ratio_thresholds[-1]:.2f}, step=0.05')
    print('=' * 80)
    print(pivot_best_hr.round(2).to_string())

    print("\n" + '=' * 80)
    print('BEST SPEARMAN THRESHOLD SELECTED PER WINDOW x HOLD CELL')
    print(f'  What it shows : Which Spearman threshold produced the best robust_score for each cell above')
    print(f'  Rows          : Rolling window size (bars) — tested: {test_windows}')
    print(f'  Columns       : Hold horizon (minutes)     — tested: {hold_horizons}')
    print(f'  Threshold range swept : {spearman_thresholds[0]:.2f} – {spearman_thresholds[-1]:.2f}, step=0.10')
    print('=' * 80)
    print(pivot_best_sp.round(2).to_string())

    print("\n" + '=' * 80)
    print('BEST ROBUSTNESS SCORE BY WINDOW x HOLD')
    print(f'  What it shows : Composite score penalising IS→OOS degradation in win rate and avg bps')
    print(f'  Rows          : Rolling window size (bars) — tested: {test_windows}')
    print(f'  Columns       : Hold horizon (minutes)     — tested: {hold_horizons}')
    print(f'  Higher = less overfitting, more stable across IS/OOS split')
    print('=' * 80)
    print(pivot_robust_score.round(2).to_string())

    fig = px.scatter(
        analysis_results,
        x='oos_win_rate_pct',
        y='oos_avg_bps',
        color='window_bars',
        size='total_events',
        hover_data=['hold_minutes', 'hit_ratio_threshold', 'spearman_threshold', 'robust_score'],
        title='Credible Parameter Sets: OOS Win Rate vs OOS Avg BPS'
    )
    fig.add_vline(x=50, line_dash='dash', line_color='white')
    fig.add_hline(y=0, line_dash='dash', line_color='white')
    fig.update_layout(height=650)
    fig.show()
else:
    print('No parameter sets were produced by the robustness search.')

# ── WALK-FORWARD: Monthly rolling validation ──────────────────────────────
print(f'\nRunning monthly walk-forward validation...')
print(f'  Same parameter grid: {len(test_windows)*len(hold_horizons)*len(hit_ratio_thresholds)*len(spearman_thresholds):,} combos')
print(f'  Method: train on all months up to month M, test on month M+1')
print(f'  Filters: min_train_months=4, min_train_events=40, min_test_events=10, min_valid_folds=6')
wf_summary, wf_folds = run_walk_forward_grid_search(
    df_ny,
    test_windows=test_windows,
    hold_horizons=hold_horizons,
    hit_ratio_thresholds=hit_ratio_thresholds,
    spearman_thresholds=spearman_thresholds,
    session_start=SIGNAL_SESSION_START,
    session_end=SIGNAL_SESSION_END,
    min_train_months=4,
    min_train_events=40,
    min_test_events=10,
    min_valid_folds=6,
)

wf_credible = wf_summary[wf_summary['passes_walk_forward']].copy()
wf_analysis = wf_credible.copy()
if wf_analysis.empty:
    print('No configurations passed the walk-forward filter. Falling back to all walk-forward results.')
    wf_analysis = wf_summary.copy()

print("\n" + '=' * 80)
print('MONTHLY WALK-FORWARD SUMMARY')
print('=' * 80)
print(f'  Total walk-forward combos evaluated             : {len(wf_summary):,}')
print(f'  Combos passing the walk-forward fold filter     : {len(wf_credible):,}')
if not wf_analysis.empty:
    print(f'  Best walk-forward score in displayed set        : {wf_analysis["walk_forward_score"].max():.2f}')
    print(f'  Best weighted OOS avg bps in displayed set      : {wf_analysis["weighted_oos_avg_bps"].max():.2f}')

wf_cols = [
    'window_bars', 'hold_minutes', 'hit_ratio_threshold', 'spearman_threshold',
    'valid_folds', 'fold_coverage_pct', 'total_test_events',
    'weighted_oos_win_rate_pct', 'weighted_oos_avg_bps',
    'positive_month_pct', 'oos_bps_std', 'mean_win_rate_gap', 'walk_forward_score'
]

if not wf_analysis.empty:
    print("\nTop 100 walk-forward configurations (ranked by repeated OOS consistency):")
    display(wf_analysis[wf_cols].head(100).round(2))

    wf_best_by_window_hold = (
        wf_analysis
        .sort_values(['walk_forward_score', 'weighted_oos_avg_bps', 'positive_month_pct'], ascending=False)
        .groupby(['window_bars', 'hold_minutes'], as_index=False)
        .first()
    )

    wf_pivot_bps     = wf_best_by_window_hold.pivot(index='window_bars', columns='hold_minutes', values='weighted_oos_avg_bps')
    wf_pivot_pos     = wf_best_by_window_hold.pivot(index='window_bars', columns='hold_minutes', values='positive_month_pct')
    wf_pivot_best_hr = wf_best_by_window_hold.pivot(index='window_bars', columns='hold_minutes', values='hit_ratio_threshold')
    wf_pivot_best_sp = wf_best_by_window_hold.pivot(index='window_bars', columns='hold_minutes', values='spearman_threshold')

    print("\n" + '=' * 80)
    print('BEST WEIGHTED OOS AVG BPS BY WINDOW x HOLD (WALK-FORWARD)')
    print(f'  What it shows : Best weighted avg OOS return (bps) across all monthly folds, per Window/Hold pair')
    print(f'  Rows          : Rolling window size (bars) — tested: {test_windows}')
    print(f'  Columns       : Hold horizon (minutes)     — tested: {hold_horizons}')
    print(f'  Selection     : Per cell, the threshold combo with highest walk_forward_score was chosen (see companion tables below)')
    print(f'  Weighting     : Each fold weighted by number of test events in that month')
    print('=' * 80)
    print(wf_pivot_bps.round(2).to_string())

    print("\n" + '=' * 80)
    print('POSITIVE-MONTH RATE (%) BY WINDOW x HOLD (WALK-FORWARD)')
    print(f'  What it shows : % of monthly OOS folds with positive avg bps for each Window/Hold pair')
    print(f'  Rows          : Rolling window size (bars) — tested: {test_windows}')
    print(f'  Columns       : Hold horizon (minutes)     — tested: {hold_horizons}')
    print(f'  Interpretation: 100% = profitable in every out-of-sample month tested')
    print('=' * 80)
    print(wf_pivot_pos.round(2).to_string())

    print("\n" + '=' * 80)
    print('BEST HIT RATIO THRESHOLD SELECTED PER WINDOW x HOLD CELL (WALK-FORWARD)')
    print(f'  What it shows : Which Hit Ratio threshold produced the best walk_forward_score for each cell above')
    print(f'  Rows          : Rolling window size (bars) — tested: {test_windows}')
    print(f'  Columns       : Hold horizon (minutes)     — tested: {hold_horizons}')
    print(f'  Threshold range swept : {hit_ratio_thresholds[0]:.2f} – {hit_ratio_thresholds[-1]:.2f}, step=0.05')
    print('=' * 80)
    print(wf_pivot_best_hr.round(2).to_string())

    print("\n" + '=' * 80)
    print('BEST SPEARMAN THRESHOLD SELECTED PER WINDOW x HOLD CELL (WALK-FORWARD)')
    print(f'  What it shows : Which Spearman threshold produced the best walk_forward_score for each cell above')
    print(f'  Rows          : Rolling window size (bars) — tested: {test_windows}')
    print(f'  Columns       : Hold horizon (minutes)     — tested: {hold_horizons}')
    print(f'  Threshold range swept : {spearman_thresholds[0]:.2f} – {spearman_thresholds[-1]:.2f}, step=0.10')
    print('=' * 80)
    print(wf_pivot_best_sp.round(2).to_string())

    wf_scatter = px.scatter(
        wf_analysis,
        x='positive_month_pct',
        y='weighted_oos_avg_bps',
        color='window_bars',
        size='total_test_events',
        hover_data=['hold_minutes', 'hit_ratio_threshold', 'spearman_threshold', 'walk_forward_score'],
        title='Walk-Forward Consistency: Positive Months vs Weighted OOS Avg BPS'
    )
    wf_scatter.add_vline(x=50, line_dash='dash', line_color='white')
    wf_scatter.add_hline(y=0, line_dash='dash', line_color='white')
    wf_scatter.update_layout(height=650)
    wf_scatter.show()

    best_wf = wf_analysis.iloc[0]
    print(f'\nBest walk-forward config: window={best_wf["window_bars"]} bars, hold={best_wf["hold_minutes"]} mins, ')
    print(f'  hit_ratio_threshold={best_wf["hit_ratio_threshold"]:.2f}, spearman_threshold={best_wf["spearman_threshold"]:.2f}')
    print(f'  walk_forward_score={best_wf["walk_forward_score"]:.4f}, weighted_oos_avg_bps={best_wf["weighted_oos_avg_bps"]:.2f}, positive_month_pct={best_wf["positive_month_pct"]:.1f}%')

    best_wf_folds = wf_folds[
        (wf_folds['window_bars'] == best_wf['window_bars'])
        & (wf_folds['hold_minutes'] == best_wf['hold_minutes'])
        & (wf_folds['hit_ratio_threshold'] == best_wf['hit_ratio_threshold'])
        & (wf_folds['spearman_threshold'] == best_wf['spearman_threshold'])
        & (wf_folds['fold_is_valid'])
    ].copy().sort_values('test_month')

    print("\nBest walk-forward config month-by-month path:")
    display(best_wf_folds[['test_month', 'train_events', 'test_events', 'oos_win_rate_pct', 'oos_avg_bps']].round(2))

    best_path_fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=(
            f'Best Walk-Forward Config: Monthly OOS Avg BPS (window={best_wf["window_bars"]}b, hold={best_wf["hold_minutes"]}m, HR>={best_wf["hit_ratio_threshold"]:.2f}, SP>={best_wf["spearman_threshold"]:.2f})',
            f'Best Walk-Forward Config: Monthly OOS Win Rate (window={best_wf["window_bars"]}b, hold={best_wf["hold_minutes"]}m, HR>={best_wf["hit_ratio_threshold"]:.2f}, SP>={best_wf["spearman_threshold"]:.2f})'
        )
    )
    best_path_fig.add_trace(
        go.Bar(x=best_wf_folds['test_month'], y=best_wf_folds['oos_avg_bps'], name='OOS Avg BPS', marker_color='#00CC96'),
        row=1, col=1
    )
    best_path_fig.add_trace(
        go.Scatter(x=best_wf_folds['test_month'], y=best_wf_folds['oos_win_rate_pct'], mode='lines+markers',
                   name='OOS Win Rate %', line=dict(color='#FFA15A', width=2)),
        row=2, col=1
    )
    best_path_fig.add_hline(y=0, line_dash='dash', line_color='white', row=1, col=1)
    best_path_fig.add_hline(y=50, line_dash='dash', line_color='white', row=2, col=1)
    best_path_fig.update_layout(height=800, showlegend=False)
    best_path_fig.update_yaxes(title_text='Avg BPS', row=1, col=1)
    best_path_fig.update_yaxes(title_text='Win Rate %', row=2, col=1)
    best_path_fig.show()
else:
    print('No parameter sets were produced by the walk-forward search.')


### Unified Robustness Score (Asymmetric Gap Penalty)

**Key Improvement:** The scoring function now uses **asymmetric gap penalties** that reward out-of-sample outperformance while still heavily penalizing overfitting.

**How it works:**
- **Overfitting Penalty** (IS > OOS): Strong penalty (−0.60× for win rate, −0.08× for returns)
- **Positive Surprise Bonus** (OOS > IS): Light reward (+0.15× for win rate, +0.08× for returns)
- **Result:** Parameters that generalize better (or even improve on OOS data) are ranked higher

This unified score is used across both the **70/30 robustness split** and the **monthly walk-forward validation**, ensuring consistent ranking methodology that:
1. ✓ Maintains preference for high-threshold parameters (like original robust_score)
2. ✓ Rewards OOS outperformance instead of treating it as degradation
3. ✓ Works identically for both validation approaches (no metric contradiction)

## 12. Universal Threshold Ranking

The previous analysis filtered configurations to the `>5 bps` universe **before** aggregating by threshold pair, which introduced selection bias — only looking at the threshold pairs that happened to already work.

This cell takes the opposite approach: it uses **100 % of the window × hold grid** (all 144 combinations per threshold pair) and ranks threshold pairs by how consistently they perform across every possible lookback and holding period. Window and hold are collapsed entirely — the only thing that matters is whether a given `(hit_ratio, spearman)` pair is robust.

In [ ]:
# ==========================================
# UNIVERSAL THRESHOLD RANKING
# Uses 100 % of the window × hold grid —
# no pre-filtering on bps or win rate.
# Goal: find which (hit_ratio, spearman)
# pair is most robust regardless of how
# you choose window or hold.
# ==========================================
print("\n" + "=" * 80)
print("UNIVERSAL THRESHOLD RANKING  (full window × hold grid)")
print(f"  Source        : analysis_results — all {len(analysis_results):,} (window, hold, HR, SP) combinations")
print(f"  Grid size     : {len(test_windows)} windows × {len(hold_horizons)} holds = {len(test_windows)*len(hold_horizons)} cells per threshold pair")
print(f"  HR range      : {hit_ratio_thresholds[0]:.2f} – {hit_ratio_thresholds[-1]:.2f}  ({len(hit_ratio_thresholds)} values)")
print(f"  SP range      : {spearman_thresholds[0]:.2f} – {spearman_thresholds[-1]:.2f}  ({len(spearman_thresholds)} values)")
print("=" * 80)

_TOTAL_GRID = len(test_windows) * len(hold_horizons)

# --- Build the aggregation table ---
# Aggregate ALL rows (not just credible ones) so nothing is hidden.
# Each metric answers a distinct question about the threshold pair.

_agg = (
    analysis_results
    .groupby(["hit_ratio_threshold", "spearman_threshold"], sort=False)
    .agg(
        n_combos            = ("window_bars",        "count"),
        n_credible          = ("passes_min_events",  "sum"),         # enough events on both sides
        mean_oos_avg_bps    = ("oos_avg_bps",        "mean"),        # raw average performance
        median_oos_avg_bps  = ("oos_avg_bps",        "median"),      # less sensitive to outlier combos
        mean_win_rate_pct   = ("oos_win_rate_pct",   "mean"),        # directional accuracy
        mean_robust_score   = ("robust_score",       "mean"),        # already penalises IS→OOS drop
        pct_positive_bps    = ("oos_avg_bps",        lambda s: (s > 0).mean() * 100),
        pct_above_5bps      = ("oos_avg_bps",        lambda s: (s > 5).mean() * 100),
    )
    .reset_index()
)

# Slippage-resilient rate: % of the full grid where BOTH oos_avg_bps > 5 AND win_rate > 50 %
_both = (
    analysis_results
    .assign(_p=lambda d: (d["oos_avg_bps"] > 5) & (d["oos_win_rate_pct"] > 50))
    .groupby(["hit_ratio_threshold", "spearman_threshold"])["_p"]
    .mean()
    .mul(100)
    .rename("pct_slippage_resilient")
)
universal = _agg.merge(_both, on=["hit_ratio_threshold", "spearman_threshold"])

# Derived columns
universal["coverage_pct"]  = (universal["n_combos"]    / _TOTAL_GRID              * 100).round(1)
universal["credible_pct"]  = (universal["n_credible"]  / universal["n_combos"]    * 100).round(1)

# Consensus score: mean robust_score weighted by slippage-resilient rate.
# A pair that scores high on average AND passes the slippage filter in many cells wins.
universal["consensus_score"] = (
    universal["mean_robust_score"] * (universal["pct_slippage_resilient"] / 100)
)

universal = universal.sort_values("consensus_score", ascending=False).reset_index(drop=True)
universal.index += 1  # 1-based rank

_display_cols = [
    "hit_ratio_threshold", "spearman_threshold",
    "n_combos", "coverage_pct", "credible_pct",
    "pct_positive_bps", "pct_above_5bps", "pct_slippage_resilient",
    "mean_oos_avg_bps", "median_oos_avg_bps",
    "mean_win_rate_pct", "mean_robust_score", "consensus_score",
]

print("\n  Column guide:")
print("    coverage_pct          → % of the 144-cell grid that produced results")
print("    credible_pct          → % of those combos with enough events for IS/OOS split")
print("    pct_positive_bps      → % of all combos with OOS avg bps > 0")
print("    pct_above_5bps        → % of all combos with OOS avg bps > 5")
print("    pct_slippage_resilient→ % of all combos with OOS avg bps > 5 AND win rate > 50 %")
print("    mean_robust_score     → avg of unified score (rewards OOS surprise, penalises overfit)")
print("    consensus_score       → mean_robust_score × slippage_resilient_rate  [SORT KEY]")
print()

print(universal[_display_cols].round(2).to_string())

_best = universal.iloc[0]
print(
    f"\n  ★ Best universal threshold pair\n"
    f"     hit_ratio  ≥ {_best['hit_ratio_threshold']:.2f}\n"
    f"     spearman   ≥ {_best['spearman_threshold']:.2f}\n"
    f"     Slippage-resilient in {_best['pct_slippage_resilient']:.1f} % of all {_TOTAL_GRID} window×hold cells\n"
    f"     Mean OOS avg bps     : {_best['mean_oos_avg_bps']:.2f}\n"
    f"     Mean win rate        : {_best['mean_win_rate_pct']:.1f} %\n"
    f"     Consensus score      : {_best['consensus_score']:.4f}"
)


# ==========================================
# HEATMAPS
# Three views of the threshold space:
#   1. Mean OOS avg bps (raw performance)
#   2. % slippage-resilient (robustness)
#   3. Consensus score (combined)
# ==========================================
def _pivot(col: str) -> pd.DataFrame:
    return universal.pivot(
        index="hit_ratio_threshold",
        columns="spearman_threshold",
        values=col,
    )

_pivots = {
    "Mean OOS Avg BPS (all combos)":                     _pivot("mean_oos_avg_bps"),
    "Slippage-Resilient Rate % (OOS bps>5 & WR>50 %)":  _pivot("pct_slippage_resilient"),
    "Consensus Score (robust score × resilience rate)":  _pivot("consensus_score"),
}

fig_heat = make_subplots(
    rows=1, cols=3,
    subplot_titles=list(_pivots.keys()),
    horizontal_spacing=0.10,
)

_cmaps = ["RdYlGn", "Blues", "Oranges"]
for col_idx, (title, pivot_df) in enumerate(_pivots.items(), start=1):
    z    = pivot_df.values.tolist()
    x    = [f"SP≥{v:.2f}" for v in pivot_df.columns.tolist()]
    y    = [f"HR≥{v:.2f}" for v in pivot_df.index.tolist()]
    text = [[f"{v:.1f}" if v == v else "—" for v in row] for row in pivot_df.values]

    fig_heat.add_trace(
        go.Heatmap(
            z=z, x=x, y=y,
            text=text,
            texttemplate="%{text}",
            colorscale=_cmaps[col_idx - 1],
            showscale=True,
            hovertemplate=f"<b>{title}</b><br>%{{y}}  %{{x}}<br>Value: %{{z:.2f}}<extra></extra>",
        ),
        row=1, col=col_idx,
    )

fig_heat.update_layout(
    title=dict(
        text=(
            "Universal Threshold Heatmaps — 100 % of Window × Hold Grid<br>"
            "<sup>Rows = Hit Ratio threshold · Columns = Spearman threshold</sup>"
        ),
        font=dict(size=14),
    ),
    height=520,
    template="plotly_dark",
)
fig_heat.show()


# ==========================================
# SCATTER: performance vs robustness
# Each point = one (HR, SP) threshold pair.
# Position → raw performance vs % resilient
# Size     → mean robust score
# ==========================================
fig_scatter = px.scatter(
    universal.reset_index(names="rank"),
    x="pct_slippage_resilient",
    y="mean_oos_avg_bps",
    size="mean_robust_score",
    color="consensus_score",
    color_continuous_scale="RdYlGn",
    text=universal.reset_index(names="rank").apply(
        lambda r: f"HR{r['hit_ratio_threshold']:.2f}/SP{r['spearman_threshold']:.2f}", axis=1
    ),
    hover_data={
        "hit_ratio_threshold": True,
        "spearman_threshold": True,
        "pct_slippage_resilient": ":.1f",
        "mean_oos_avg_bps": ":.2f",
        "mean_robust_score": ":.3f",
        "consensus_score": ":.4f",
    },
    title=(
        "Threshold Pair Quality Map — Performance vs Robustness<br>"
        "<sup>X = % of grid cells that are slippage-resilient · "
        "Y = mean OOS avg bps · Size = mean robust score · Colour = consensus score</sup>"
    ),
)
fig_scatter.update_traces(textposition="top center", textfont_size=9)
fig_scatter.add_vline(x=0, line_dash="dash", line_color="rgba(255,255,255,0.3)")
fig_scatter.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.3)")
fig_scatter.update_layout(height=600, template="plotly_dark")
fig_scatter.show()


## 13. NY Session Comparison

The 24-hour robustness search included all market hours including overnight and pre-market. This cell re-runs the identical search restricted to **09:30–16:00 ET (Regular Trading Hours)** only, then produces a direct side-by-side comparison.

Key questions this answers:
- Does the BTC→NQ regime signal hold up in liquid hours, or is it driven by thin overnight moves?
- Do the optimal threshold pairs change when we restrict to RTH?
- How much of the edge is session-specific vs truly cross-session?

In [ ]:
# ==========================================
# NY SESSION UNIVERSAL THRESHOLD RANKING
# Reruns the robustness search restricted
# to 09:30–16:00 ET, then compares results
# against the full 24-hour session.
# ==========================================

# ------------------------------------------------------------------
# Helper: build universal threshold table + heatmaps + scatter
# for any analysis_results dataframe.
# ------------------------------------------------------------------
def _universal_threshold_ranking(
    results:       pd.DataFrame,
    session_label: str,
    total_grid:    int,
) -> pd.DataFrame:
    """
    Aggregate analysis_results by (hit_ratio, spearman) only,
    produce the ranked table and three visualisations, and
    return the ranked universal DataFrame.
    """
    _agg = (
        results
        .groupby(["hit_ratio_threshold", "spearman_threshold"], sort=False)
        .agg(
            n_combos            = ("window_bars",       "count"),
            n_credible          = ("passes_min_events", "sum"),
            mean_oos_avg_bps    = ("oos_avg_bps",       "mean"),
            median_oos_avg_bps  = ("oos_avg_bps",       "median"),
            mean_win_rate_pct   = ("oos_win_rate_pct",  "mean"),
            mean_robust_score   = ("robust_score",      "mean"),
            pct_positive_bps    = ("oos_avg_bps",       lambda s: (s > 0).mean() * 100),
            pct_above_5bps      = ("oos_avg_bps",       lambda s: (s > 5).mean() * 100),
        )
        .reset_index()
    )

    _both = (
        results
        .assign(_p=lambda d: (d["oos_avg_bps"] > 5) & (d["oos_win_rate_pct"] > 50))
        .groupby(["hit_ratio_threshold", "spearman_threshold"])["_p"]
        .mean()
        .mul(100)
        .rename("pct_slippage_resilient")
    )
    uni = _agg.merge(_both, on=["hit_ratio_threshold", "spearman_threshold"])
    uni["coverage_pct"]    = (uni["n_combos"]   / total_grid              * 100).round(1)
    uni["credible_pct"]    = (uni["n_credible"] / uni["n_combos"]         * 100).round(1)
    uni["consensus_score"] = uni["mean_robust_score"] * (uni["pct_slippage_resilient"] / 100)
    uni = uni.sort_values("consensus_score", ascending=False).reset_index(drop=True)
    uni.index += 1

    # --- Table ---
    _cols = [
        "hit_ratio_threshold", "spearman_threshold",
        "n_combos", "coverage_pct", "credible_pct",
        "pct_positive_bps", "pct_above_5bps", "pct_slippage_resilient",
        "mean_oos_avg_bps", "median_oos_avg_bps",
        "mean_win_rate_pct", "mean_robust_score", "consensus_score",
    ]
    print(f"\n  Universal threshold table — {session_label}")
    print(uni[_cols].round(2).to_string())

    _best = uni.iloc[0]
    print(
        f"\n  ★ Best pair ({session_label})\n"
        f"     hit_ratio  ≥ {_best['hit_ratio_threshold']:.2f}  |  "
        f"spearman   ≥ {_best['spearman_threshold']:.2f}\n"
        f"     Slippage-resilient in {_best['pct_slippage_resilient']:.1f} % "
        f"of all {total_grid} window×hold cells\n"
        f"     Mean OOS avg bps  : {_best['mean_oos_avg_bps']:.2f}\n"
        f"     Mean win rate     : {_best['mean_win_rate_pct']:.1f} %\n"
        f"     Consensus score   : {_best['consensus_score']:.4f}"
    )

    # --- Heatmaps ---
    def _pivot(col):
        return uni.pivot(
            index="hit_ratio_threshold",
            columns="spearman_threshold",
            values=col,
        )

    _pivots = {
        "Mean OOS Avg BPS":           _pivot("mean_oos_avg_bps"),
        "Slippage-Resilient Rate %":  _pivot("pct_slippage_resilient"),
        "Consensus Score":            _pivot("consensus_score"),
    }
    fig_heat = make_subplots(
        rows=1, cols=3,
        subplot_titles=[f"{t}<br><sup>{session_label}</sup>" for t in _pivots],
        horizontal_spacing=0.10,
    )
    for col_idx, (_, pivot_df) in enumerate(_pivots.items(), start=1):
        z    = pivot_df.values.tolist()
        x    = [f"SP≥{v:.2f}" for v in pivot_df.columns.tolist()]
        y    = [f"HR≥{v:.2f}" for v in pivot_df.index.tolist()]
        text = [[f"{v:.1f}" if v == v else "—" for v in row] for row in pivot_df.values]
        fig_heat.add_trace(
            go.Heatmap(
                z=z, x=x, y=y,
                text=text, texttemplate="%{text}",
                colorscale=["RdYlGn", "Blues", "Oranges"][col_idx - 1],
                showscale=True,
                hovertemplate="%{y}  %{x}<br>Value: %{z:.2f}<extra></extra>",
            ),
            row=1, col=col_idx,
        )
    fig_heat.update_layout(
        title=dict(
            text=(
                f"Universal Threshold Heatmaps — {session_label}<br>"
                "<sup>Rows = Hit Ratio · Columns = Spearman · "
                "100 % of window × hold grid</sup>"
            ),
            font=dict(size=14),
        ),
        height=500,
        template="plotly_dark",
    )
    fig_heat.show()

    # --- Scatter: performance vs robustness ---
    _scatter_df = uni.reset_index(names="rank")
    _scatter_df["label"] = _scatter_df.apply(
        lambda r: f"HR{r['hit_ratio_threshold']:.2f}/SP{r['spearman_threshold']:.2f}", axis=1
    )
    fig_scatter = px.scatter(
        _scatter_df,
        x="pct_slippage_resilient",
        y="mean_oos_avg_bps",
        size="mean_robust_score",
        color="consensus_score",
        color_continuous_scale="RdYlGn",
        text="label",
        hover_data={
            "hit_ratio_threshold":    True,
            "spearman_threshold":     True,
            "pct_slippage_resilient": ":.1f",
            "mean_oos_avg_bps":       ":.2f",
            "consensus_score":        ":.4f",
        },
        title=(
            f"Threshold Pair Quality Map — {session_label}<br>"
            "<sup>X = % slippage-resilient · Y = mean OOS avg bps · "
            "Size = mean robust score · Colour = consensus score</sup>"
        ),
    )
    fig_scatter.update_traces(textposition="top center", textfont_size=9)
    fig_scatter.add_vline(x=0, line_dash="dash", line_color="rgba(255,255,255,0.3)")
    fig_scatter.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.3)")
    fig_scatter.update_layout(height=580, template="plotly_dark")
    fig_scatter.show()

    return uni


# ------------------------------------------------------------------
# Re-run the robustness search for NY session only
# ------------------------------------------------------------------
NY_SESSION_START = "09:30"
NY_SESSION_END   = "16:00"

print("=" * 80)
print("NY SESSION ROBUSTNESS SEARCH  (09:30 – 16:00 ET)")
print(f"  Same parameter grid as 24-hour search:")
print(f"  {len(test_windows)} windows × {len(hold_horizons)} holds × "
      f"{len(hit_ratio_thresholds)} HR × {len(spearman_thresholds)} SP "
      f"= {len(test_windows)*len(hold_horizons)*len(hit_ratio_thresholds)*len(spearman_thresholds):,} combos")
print(f"  IS/OOS split : 70 % / 30 %  |  Min events per side : {min_events_per_side}")
print("=" * 80)

robust_results_ny = run_robustness_grid_search(
    df_ny,
    test_windows          = test_windows,
    hold_horizons         = hold_horizons,
    hit_ratio_thresholds  = hit_ratio_thresholds,
    spearman_thresholds   = spearman_thresholds,
    session_start         = NY_SESSION_START,
    session_end           = NY_SESSION_END,
    split_ratio           = 0.70,
    min_events_per_side   = min_events_per_side,
)

analysis_results_ny = robust_results_ny[robust_results_ny["passes_min_events"]].copy()
if analysis_results_ny.empty:
    print("  No configurations passed the min-event filter — using all results.")
    analysis_results_ny = robust_results_ny.copy()

print(f"\n  Total combos evaluated      : {len(robust_results_ny):,}")
print(f"  Combos passing event filter : {len(analysis_results_ny):,}")
print(f"  Best OOS avg bps            : {analysis_results_ny['oos_avg_bps'].max():.2f}")
print(f"  Best robust score           : {analysis_results_ny['robust_score'].max():.2f}")


# ------------------------------------------------------------------
# Universal ranking — NY session
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("UNIVERSAL THRESHOLD RANKING  —  NY SESSION (09:30 – 16:00 ET)")
print("=" * 80)

_TOTAL_GRID = len(test_windows) * len(hold_horizons)

universal_ny = _universal_threshold_ranking(
    results       = analysis_results_ny,
    session_label = "NY Session (09:30–16:00)",
    total_grid    = _TOTAL_GRID,
)


# ------------------------------------------------------------------
# Side-by-side comparison: 24h vs NY session
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("SESSION COMPARISON  —  24-Hour  vs  NY Session (09:30–16:00)")
print("  Same (hit_ratio, spearman) pairs, same methodology.")
print("  Differences reveal how much of the signal is session-specific.")
print("=" * 80)

# Rebuild 24h ranking using the same function so columns are guaranteed identical
universal_24h = _universal_threshold_ranking(
    results       = analysis_results,
    session_label = "24-Hour",
    total_grid    = _TOTAL_GRID,
)

# Merge on threshold pair
_MERGE_KEYS = ["hit_ratio_threshold", "spearman_threshold"]
_COMPARE_COLS = [
    "mean_oos_avg_bps", "pct_slippage_resilient",
    "mean_win_rate_pct", "mean_robust_score", "consensus_score",
]

comparison = universal_24h[_MERGE_KEYS + _COMPARE_COLS].merge(
    universal_ny[_MERGE_KEYS + _COMPARE_COLS],
    on=_MERGE_KEYS,
    suffixes=("_24h", "_ny"),
)

# Deltas: NY minus 24h (positive = NY is better for that metric)
for col in _COMPARE_COLS:
    comparison[f"Δ_{col}"] = comparison[f"{col}_ny"] - comparison[f"{col}_24h"]

comparison = comparison.sort_values("consensus_score_ny", ascending=False).reset_index(drop=True)
comparison.index += 1

_comp_display = (
    _MERGE_KEYS
    + [f"{c}_24h" for c in _COMPARE_COLS]
    + [f"{c}_ny"  for c in _COMPARE_COLS]
    + [f"Δ_{c}"   for c in ["mean_oos_avg_bps", "pct_slippage_resilient", "consensus_score"]]
)

print("\n  Δ columns = NY minus 24h  |  positive = NY session performs better\n")
print(comparison[_comp_display].round(2).to_string())

# --- Delta bar chart ---
_top_pairs = comparison.head(10).copy()
_top_pairs["label"] = _top_pairs.apply(
    lambda r: f"HR{r['hit_ratio_threshold']:.2f}/SP{r['spearman_threshold']:.2f}", axis=1
)

fig_delta = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Δ Mean OOS Avg BPS<br><sup>(NY minus 24h)</sup>",
        "Δ Slippage-Resilient %<br><sup>(NY minus 24h)</sup>",
        "Δ Consensus Score<br><sup>(NY minus 24h)</sup>",
    ],
    horizontal_spacing=0.12,
)

for col_idx, delta_col in enumerate(
    ["Δ_mean_oos_avg_bps", "Δ_pct_slippage_resilient", "Δ_consensus_score"], start=1
):
    vals   = _top_pairs[delta_col].tolist()
    colors = ["#00CC96" if v >= 0 else "#EF553B" for v in vals]
    fig_delta.add_trace(
        go.Bar(
            x=_top_pairs["label"].tolist(),
            y=vals,
            marker_color=colors,
            hovertemplate=f"<b>%{{x}}</b><br>{delta_col}: %{{y:.2f}}<extra></extra>",
        ),
        row=1, col=col_idx,
    )
    fig_delta.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.4)",
                        row=1, col=col_idx)

fig_delta.update_layout(
    title=dict(
        text=(
            "NY Session vs 24-Hour — Delta per Threshold Pair  "
            "(top 10 NY pairs by consensus score)<br>"
            "<sup>Green = NY outperforms · Red = 24h outperforms</sup>"
        ),
        font=dict(size=13),
    ),
    height=480,
    template="plotly_dark",
    showlegend=False,
)
fig_delta.update_xaxes(tickangle=30)
fig_delta.show()

# --- Summary printout ---
_best_24h = universal_24h.iloc[0]
_best_ny  = universal_ny.iloc[0]

print("\n  SUMMARY")
print(f"  {'':30s}  {'24-Hour':>20s}  {'NY Session':>20s}")
print(f"  {'-'*72}")
print(f"  {'Best HR threshold':30s}  {_best_24h['hit_ratio_threshold']:>20.2f}  {_best_ny['hit_ratio_threshold']:>20.2f}")
print(f"  {'Best SP threshold':30s}  {_best_24h['spearman_threshold']:>20.2f}  {_best_ny['spearman_threshold']:>20.2f}")
print(f"  {'Slippage-resilient %':30s}  {_best_24h['pct_slippage_resilient']:>20.1f}  {_best_ny['pct_slippage_resilient']:>20.1f}")
print(f"  {'Mean OOS avg bps':30s}  {_best_24h['mean_oos_avg_bps']:>20.2f}  {_best_ny['mean_oos_avg_bps']:>20.2f}")
print(f"  {'Mean win rate %':30s}  {_best_24h['mean_win_rate_pct']:>20.1f}  {_best_ny['mean_win_rate_pct']:>20.1f}")
print(f"  {'Consensus score':30s}  {_best_24h['consensus_score']:>20.4f}  {_best_ny['consensus_score']:>20.4f}")


## 14. Deep Analysis: HR ≥ 0.80 | SP ≥ 0.70 | NY Session

With the best universal threshold pair identified, this cell focuses exclusively on **HR ≥ 0.80 and SP ≥ 0.70 in the NY session**.

### Window & hold selection methodology
Rather than simply picking the (window, hold) cell with the highest OOS avg bps — which risks selecting an over-fitted spike — each cell is scored on its **neighbourhood robustness**: the fraction of its 3×3 grid neighbourhood (itself plus up to 8 adjacent cells) that also passes the slippage-resilient filter (OOS avg bps > 5 AND win rate > 50%).

A cell at the peak of a genuine profitable plateau scores close to 1.0 because all its neighbours also pass. An isolated spike scores ≤ 1/9 ≈ 0.11 even if its own bps is the highest on the entire surface.

### Equity curve analysis
For each selected (window, hold) pair the full trade sequence is reconstructed from scratch using the raw indicator columns, and a six-panel dashboard is produced covering: cumulative PnL, drawdown, rolling win rate, monthly PnL bars, return distribution, and rolling average bps.

In [ ]:
# ==========================================
# DEEP ANALYSIS: HR ≥ 0.80 | SP ≥ 0.70 | NY SESSION
# ==========================================
# Goal:
#   1. Map the full (window, hold) performance surface for this threshold pair.
#   2. Select windows/holds from a profitable plateau — not isolated spikes —
#      by scoring each cell on how many of its grid neighbours also pass
#      the slippage-resilient filter.
#   3. Run a thorough equity-curve analysis on the selected configurations.
# ==========================================
from scipy.ndimage import uniform_filter

BEST_HR   = 0.80
BEST_SP   = 0.70
NY_START  = "09:30"
NY_END    = "16:00"

print("=" * 80)
print(f"DEEP ANALYSIS  |  HR ≥ {BEST_HR}  |  SP ≥ {BEST_SP}  |  NY Session ({NY_START}–{NY_END})")
print("=" * 80)


# ──────────────────────────────────────────────────────────────────
# 1. PERFORMANCE SURFACE
#    Extract the (window, hold) grid for the fixed threshold pair.
# ──────────────────────────────────────────────────────────────────
_surface = analysis_results_ny[
    np.isclose(analysis_results_ny["hit_ratio_threshold"], BEST_HR) &
    np.isclose(analysis_results_ny["spearman_threshold"],  BEST_SP)
].copy()

if _surface.empty:
    raise RuntimeError(
        f"No rows found for HR={BEST_HR}, SP={BEST_SP} in analysis_results_ny. "
        "Check that the NY session search ran with these threshold values."
    )

print(f"\n  Surface cells found : {len(_surface)}  "
      f"(expected ≤ {len(test_windows) * len(hold_horizons)})")

_pivot_bps = _surface.pivot(index="window_bars", columns="hold_minutes", values="oos_avg_bps")
_pivot_wr  = _surface.pivot(index="window_bars", columns="hold_minutes", values="oos_win_rate_pct")
_pivot_rs  = _surface.pivot(index="window_bars", columns="hold_minutes", values="robust_score")
_pivot_ev  = _surface.pivot(index="window_bars", columns="hold_minutes", values="total_events")

# Surface heatmaps
fig_surface = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "OOS Avg BPS",
        "OOS Win Rate %",
        "Robust Score",
    ],
    horizontal_spacing=0.10,
)
for col_idx, (pivot_df, cmap, fmt) in enumerate([
    (_pivot_bps, "RdYlGn", ".1f"),
    (_pivot_wr,  "RdYlGn", ".1f"),
    (_pivot_rs,  "Blues",  ".2f"),
], start=1):
    z    = pivot_df.values.tolist()
    x    = [f"{int(c)}m" for c in pivot_df.columns]
    y    = [f"{int(r)}b" for r in pivot_df.index]
    text = [[f"{v:{fmt}}" if v == v else "—" for v in row] for row in pivot_df.values]
    fig_surface.add_trace(
        go.Heatmap(
            z=z, x=x, y=y,
            text=text, texttemplate="%{text}",
            colorscale=cmap, showscale=True,
            hovertemplate="Window %{y}  Hold %{x}<br>Value: %{z:.2f}<extra></extra>",
        ),
        row=1, col=col_idx,
    )
fig_surface.update_layout(
    title=dict(
        text=(
            f"Performance Surface — HR ≥ {BEST_HR}  |  SP ≥ {BEST_SP}  |  NY Session<br>"
            "<sup>Rows = rolling window (bars) · Columns = hold horizon (minutes)</sup>"
        ),
        font=dict(size=14),
    ),
    height=480,
    template="plotly_dark",
)
fig_surface.show()


# ──────────────────────────────────────────────────────────────────
# 2. NEIGHBOURHOOD ROBUSTNESS  (plateau detection)
#    For every (window, hold) cell, compute what fraction of its
#    3×3 grid neighbourhood also passes the slippage-resilient
#    filter (OOS avg bps > 5 AND win rate > 50 %).
#    A cell surrounded by neighbours that all pass scores 1.0;
#    an isolated spike that passes alone scores ≤ 1/9 ≈ 0.11.
# ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PLATEAU DETECTION  —  Neighbourhood Robustness")
print("  Method : 3×3 sliding window over the binary pass matrix.")
print("  Score  : fraction of the 9 cells (self + 8 neighbours) passing")
print("           OOS avg bps > 5  AND  OOS win rate > 50 %.")
print("  Goal   : isolate a profitable plateau, not an over-fitted spike.")
print("=" * 80)

# Binary pass matrix aligned to the full pivot grid
_pass_matrix = ((_pivot_bps > 5) & (_pivot_wr > 50)).reindex(
    index=_pivot_bps.index, columns=_pivot_bps.columns
).fillna(False).astype(float)

_nb_scores_arr = uniform_filter(
    _pass_matrix.values.astype(float),
    size=3,
    mode="constant",
    cval=0.0,
)
_nb_scores = pd.DataFrame(
    _nb_scores_arr,
    index=_pivot_bps.index,
    columns=_pivot_bps.columns,
)

# Attach neighbourhood score back to the surface dataframe
_surface["nb_score"] = _surface.apply(
    lambda r: _nb_scores.loc[r["window_bars"], r["hold_minutes"]]
    if (r["window_bars"] in _nb_scores.index and r["hold_minutes"] in _nb_scores.columns)
    else 0.0,
    axis=1,
)

# Neighbourhood score heatmap
fig_nb = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Pass Matrix (OOS bps>5 & WR>50%)",
        "Neighbourhood Robustness Score",
    ],
    horizontal_spacing=0.12,
)
for col_idx, (mat, cmap) in enumerate([
    (_pass_matrix, "Blues"),
    (_nb_scores,   "RdYlGn"),
], start=1):
    z    = mat.values.tolist()
    x    = [f"{int(c)}m" for c in mat.columns]
    y    = [f"{int(r)}b" for r in mat.index]
    text = [[f"{v:.2f}" for v in row] for row in mat.values]
    fig_nb.add_trace(
        go.Heatmap(
            z=z, x=x, y=y,
            text=text, texttemplate="%{text}",
            colorscale=cmap, showscale=True,
            hovertemplate="Window %{y}  Hold %{x}<br>Score: %{z:.2f}<extra></extra>",
        ),
        row=1, col=col_idx,
    )
fig_nb.update_layout(
    title=dict(
        text=(
            "Plateau Detection — Neighbourhood Robustness<br>"
            "<sup>Left: 1 = passes slippage filter · Right: fraction of 3×3 neighbours passing</sup>"
        ),
        font=dict(size=14),
    ),
    height=450,
    template="plotly_dark",
)
fig_nb.show()


# ──────────────────────────────────────────────────────────────────
# 3. PLATEAU CANDIDATE SELECTION
#    Keep cells at the top neighbourhood score tier, then rank by
#    OOS avg bps as a secondary criterion.  Pick up to 5.
# ──────────────────────────────────────────────────────────────────
_max_nb  = _surface["nb_score"].max()
_plateau = _surface[
    _surface["nb_score"] >= _max_nb - 1e-9    # cells at the peak neighbourhood tier
].sort_values(["nb_score", "oos_avg_bps"], ascending=[False, False])

N_SELECTED  = min(5, len(_plateau))
_selected   = _plateau.head(N_SELECTED).copy()

print(f"\n  Peak neighbourhood score   : {_max_nb:.3f}  "
      f"({'%.0f' % (_max_nb * 9)}/9 neighbours pass)")
print(f"  Cells at peak tier         : {len(_plateau)}")
print(f"  Selected for deep analysis : {N_SELECTED}\n")

_sel_display = [
    "window_bars", "hold_minutes",
    "total_events", "oos_win_rate_pct", "oos_avg_bps",
    "robust_score", "nb_score",
]
display(_selected[_sel_display].round(2).reset_index(drop=True))


# ──────────────────────────────────────────────────────────────────
# 4. TRADE-LEVEL EQUITY CURVE BUILDER
#    Re-derives signals from the raw indicator columns for NY hours
#    so each (window, hold) pair is evaluated independently.
# ──────────────────────────────────────────────────────────────────
def _build_trades(
    df:           pd.DataFrame,
    window:       int,
    hold_minutes: int,
    hr_threshold: float,
    sp_threshold: float,
    session_start: str,
    session_end:   str,
) -> pd.DataFrame:
    """
    Re-compute rolling indicators for a specific window, apply the
    session + threshold filters, and return one row per trade with:
      - timestamp, forward_bps, cumulative_bps, drawdown_bps
    """
    tmp = df[["btc_ret", "nq_ret", "nq_close"]].copy()

    # Rolling indicators for this specific window
    from src.indicators import fast_rolling_spearman
    tmp["corr_sp"]  = fast_rolling_spearman(
        tmp["btc_ret"].to_numpy(),
        tmp["nq_ret"].to_numpy(),
        window,
    )
    direction_match = (np.sign(tmp["btc_ret"]) == np.sign(tmp["nq_ret"])).astype(float)
    tmp["hit_ratio"] = direction_match.rolling(window).mean()
    tmp = tmp.dropna(subset=["corr_sp", "hit_ratio"])

    # Session filter (index is already in NY timezone)
    time_ok = (
        (tmp.index.time >= pd.to_datetime(session_start).time()) &
        (tmp.index.time <= pd.to_datetime(session_end).time())
    )
    # Threshold filter + event-start detection
    valid   = time_ok & (tmp["hit_ratio"] >= hr_threshold) & (tmp["corr_sp"] >= sp_threshold)
    ev_start = valid & (~valid.shift(1, fill_value=False))
    events  = tmp[ev_start].copy()

    if events.empty:
        return pd.DataFrame()

    # Forward return
    closes   = tmp["nq_close"].to_numpy()
    rets_arr = tmp["nq_ret"].to_numpy()
    iloc_map = {ts: i for i, ts in enumerate(tmp.index)}

    rows = []
    for ts in events.index:
        pos    = iloc_map[ts]
        future = pos + hold_minutes
        if future >= len(closes):
            continue
        direction = np.sign(rets_arr[pos]) or 1.0
        bps       = (closes[future] / closes[pos] - 1.0) * direction * 10_000
        rows.append({"timestamp": ts, "bps": bps})

    if not rows:
        return pd.DataFrame()

    trades            = pd.DataFrame(rows).set_index("timestamp")
    trades["cum_bps"] = trades["bps"].cumsum()
    running_max       = trades["cum_bps"].cummax()
    trades["dd_bps"]  = trades["cum_bps"] - running_max   # always ≤ 0
    return trades


def _equity_stats(trades: pd.DataFrame) -> dict:
    """Return a dictionary of standard equity-curve statistics."""
    bps        = trades["bps"]
    cum        = trades["cum_bps"]
    dd         = trades["dd_bps"]
    n          = len(bps)
    win_rate   = (bps > 0).mean() * 100
    avg_win    = bps[bps > 0].mean() if (bps > 0).any() else np.nan
    avg_loss   = bps[bps < 0].mean() if (bps < 0).any() else np.nan
    profit_factor = (bps[bps > 0].sum() / abs(bps[bps < 0].sum())
                     if (bps < 0).any() else np.inf)
    sharpe     = bps.mean() / bps.std() * np.sqrt(n) if bps.std() > 0 else np.nan
    sortino    = (bps.mean() / bps[bps < 0].std() * np.sqrt(n)
                  if (bps < 0).any() else np.nan)
    max_dd     = dd.min()
    calmar     = (cum.iloc[-1] / abs(max_dd)) if max_dd < 0 else np.inf
    # Longest drawdown duration (in trades)
    in_dd      = (dd < 0).astype(int)
    dd_runs    = in_dd * (in_dd.groupby((in_dd != in_dd.shift()).cumsum()).cumcount() + 1)
    max_dd_dur = int(dd_runs.max())
    return dict(
        n_trades=n,
        win_rate_pct=round(win_rate, 2),
        avg_bps=round(bps.mean(), 2),
        median_bps=round(bps.median(), 2),
        avg_win_bps=round(avg_win, 2),
        avg_loss_bps=round(avg_loss, 2),
        profit_factor=round(profit_factor, 3),
        sharpe=round(sharpe, 3),
        sortino=round(sortino, 3),
        total_bps=round(cum.iloc[-1], 1),
        max_dd_bps=round(max_dd, 1),
        calmar=round(calmar, 3),
        max_dd_duration_trades=max_dd_dur,
    )


# ──────────────────────────────────────────────────────────────────
# 5. EQUITY CURVE ANALYSIS  (one section per selected config)
# ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("EQUITY CURVE ANALYSIS  —  Selected Plateau Configurations")
print("=" * 80)

all_stats   = []
all_trades  = {}

for _, cfg in _selected.iterrows():
    W = int(cfg["window_bars"])
    H = int(cfg["hold_minutes"])
    label = f"W={W}b  H={H}m"

    trades = _build_trades(
        df            = df_ny,
        window        = W,
        hold_minutes  = H,
        hr_threshold  = BEST_HR,
        sp_threshold  = BEST_SP,
        session_start = NY_START,
        session_end   = NY_END,
    )

    if trades.empty:
        print(f"  [{label}]  No trades generated — skipping.")
        continue

    all_trades[label] = trades
    stats = _equity_stats(trades)
    stats["config"] = label
    all_stats.append(stats)

    # ── Per-config full dashboard ──────────────────────────────────
    print(f"\n{'─'*60}")
    print(f"  CONFIG : {label}  |  "
          f"HR ≥ {BEST_HR}  SP ≥ {BEST_SP}  |  NY Session")
    print(f"{'─'*60}")

    fig_cfg = make_subplots(
        rows=3, cols=2,
        subplot_titles=[
            "Cumulative PnL (bps)",
            "Drawdown (bps)",
            "Rolling 50-Trade Win Rate (%)",
            "Monthly PnL (bps)",
            "Trade Return Distribution",
            "Rolling 50-Trade Avg BPS",
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.10,
    )

    # 1. Cumulative PnL
    fig_cfg.add_trace(
        go.Scatter(
            x=list(range(len(trades))),
            y=trades["cum_bps"].tolist(),
            mode="lines",
            name="Cum PnL",
            line=dict(color="#00CC96", width=2),
        ),
        row=1, col=1,
    )
    fig_cfg.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.3)", row=1, col=1)

    # 2. Drawdown
    fig_cfg.add_trace(
        go.Scatter(
            x=list(range(len(trades))),
            y=trades["dd_bps"].tolist(),
            mode="lines",
            name="Drawdown",
            line=dict(color="#EF553B", width=1.5),
            fill="tozeroy",
            fillcolor="rgba(239,85,59,0.15)",
        ),
        row=1, col=2,
    )

    # 3. Rolling 50-trade win rate
    _roll_wr = (trades["bps"] > 0).astype(float).rolling(50).mean() * 100
    fig_cfg.add_trace(
        go.Scatter(
            x=list(range(len(trades))),
            y=_roll_wr.tolist(),
            mode="lines",
            name="Rolling WR",
            line=dict(color="#FFA15A", width=1.5),
        ),
        row=2, col=1,
    )
    fig_cfg.add_hline(y=50, line_dash="dash", line_color="rgba(255,255,255,0.3)", row=2, col=1)

    # 4. Monthly PnL bars
    _monthly = (
        trades["bps"]
        .resample("ME")
        .sum()
        .rename("monthly_bps")
        .reset_index()
    )
    _monthly["month_str"] = _monthly["timestamp"].dt.strftime("%Y-%m")
    _m_colors = ["#00CC96" if v >= 0 else "#EF553B" for v in _monthly["monthly_bps"]]
    fig_cfg.add_trace(
        go.Bar(
            x=_monthly["month_str"].tolist(),
            y=_monthly["monthly_bps"].tolist(),
            marker_color=_m_colors,
            name="Monthly PnL",
        ),
        row=2, col=2,
    )
    fig_cfg.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.3)", row=2, col=2)

    # 5. Return distribution
    fig_cfg.add_trace(
        go.Histogram(
            x=trades["bps"].tolist(),
            nbinsx=40,
            marker_color="#636EFA",
            opacity=0.75,
            name="Distribution",
        ),
        row=3, col=1,
    )
    fig_cfg.add_vline(x=0, line_dash="dash", line_color="rgba(255,255,255,0.4)", row=3, col=1)

    # 6. Rolling 50-trade avg bps
    _roll_avg = trades["bps"].rolling(50).mean()
    fig_cfg.add_trace(
        go.Scatter(
            x=list(range(len(trades))),
            y=_roll_avg.tolist(),
            mode="lines",
            name="Rolling Avg BPS",
            line=dict(color="#AB63FA", width=1.5),
        ),
        row=3, col=2,
    )
    fig_cfg.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.3)", row=3, col=2)

    fig_cfg.update_layout(
        title=dict(
            text=(
                f"Equity Curve Dashboard — {label}  |  "
                f"HR ≥ {BEST_HR}  SP ≥ {BEST_SP}  |  NY Session<br>"
                f"<sup>{stats['n_trades']} trades · "
                f"Win rate {stats['win_rate_pct']:.1f}% · "
                f"Total PnL {stats['total_bps']:.0f} bps · "
                f"Max DD {stats['max_dd_bps']:.0f} bps · "
                f"Sharpe {stats['sharpe']:.2f}</sup>"
            ),
            font=dict(size=13),
        ),
        height=900,
        template="plotly_dark",
        showlegend=False,
    )
    fig_cfg.update_xaxes(title_text="Trade #",   row=1, col=1)
    fig_cfg.update_xaxes(title_text="Trade #",   row=1, col=2)
    fig_cfg.update_xaxes(title_text="Trade #",   row=2, col=1)
    fig_cfg.update_xaxes(title_text="Month",     row=2, col=2)
    fig_cfg.update_xaxes(title_text="BPS",       row=3, col=1)
    fig_cfg.update_xaxes(title_text="Trade #",   row=3, col=2)
    fig_cfg.update_yaxes(title_text="Cum BPS",   row=1, col=1)
    fig_cfg.update_yaxes(title_text="DD BPS",    row=1, col=2)
    fig_cfg.update_yaxes(title_text="Win Rate%", row=2, col=1)
    fig_cfg.update_yaxes(title_text="BPS",       row=2, col=2)
    fig_cfg.update_yaxes(title_text="Freq",      row=3, col=1)
    fig_cfg.update_yaxes(title_text="Avg BPS",   row=3, col=2)
    fig_cfg.show()


# ──────────────────────────────────────────────────────────────────
# 6. SIDE-BY-SIDE SUMMARY TABLE  (all selected configs)
# ──────────────────────────────────────────────────────────────────
if all_stats:
    print("\n" + "=" * 80)
    print("SUMMARY TABLE  —  All Selected Plateau Configurations")
    print("=" * 80)

    stats_df = pd.DataFrame(all_stats).set_index("config")
    print(stats_df.round(2).to_string())

    # ── Overlaid equity curves (all selected configs on one chart) ──
    fig_overlay = go.Figure()
    _palette    = px.colors.qualitative.Plotly

    for i, (lbl, trades) in enumerate(all_trades.items()):
        color = _palette[i % len(_palette)]
        fig_overlay.add_trace(go.Scatter(
            x=list(range(len(trades))),
            y=trades["cum_bps"].tolist(),
            mode="lines",
            name=lbl,
            line=dict(color=color, width=2),
            hovertemplate=f"<b>{lbl}</b><br>Trade #%{{x}}<br>Cum PnL: %{{y:.0f}} bps<extra></extra>",
        ))

    fig_overlay.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.3)")
    fig_overlay.update_layout(
        title=dict(
            text=(
                f"Overlaid Equity Curves — All Plateau Configs<br>"
                f"<sup>HR ≥ {BEST_HR}  |  SP ≥ {BEST_SP}  |  NY Session  |  "
                f"Selected from neighbourhood-robustness plateau</sup>"
            ),
            font=dict(size=13),
        ),
        xaxis_title="Trade Number",
        yaxis_title="Cumulative PnL (bps)",
        height=500,
        template="plotly_dark",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig_overlay.show()


## 15. Exit Criterion Research

With the entry mechanism locked in (`W=55b | HR≥0.80 | SP≥0.70 | NY Session`), this cell investigates whether a **dynamic exit** — one that responds to what the market is doing after entry — outperforms a fixed hold.

Two paths are compared against a **fixed 35-minute hold benchmark** (the best configuration from the plateau analysis):

**Path A — Correlation Deterioration Exit**
Monitor the rolling HR and Spearman *during* the hold. Exit at the first minute either drops below a configurable threshold. The idea is that the edge in this strategy comes from BTC and NQ moving together — once that co-movement breaks down, the reason for holding the position is gone.

**Path B — Divergence / Reversal Exit**
Two sub-mechanisms:
- *B1 (fast Spearman flip)*: compute a short rolling Spearman over live bars during the hold. Exit when it turns negative (correlation has reversed, not just weakened).
- *B2 (momentum reversal)*: exit when NQ has accumulated a threshold loss in bps against the trade direction over the last K bars.

Both paths use a hard cap of 60 minutes — no trade is ever held longer regardless of whether the exit signal fires.

In [ ]:
# ==========================================
# EXIT CRITERION RESEARCH
# Base config : W=55b | HR ≥ 0.80 | SP ≥ 0.70 | NY Session
# Benchmark   : Fixed hold of 35 minutes (best Sharpe/Sortino/Calmar from plateau)
#
# Two exit paths researched:
#   A. Correlation deterioration — exit when rolling HR or Spearman drops
#      below a configurable threshold during the hold window.
#   B. Divergence reversal      — exit when BTC and NQ start moving against
#      each other (short-window directional flip) during the hold window.
#
# Methodology:
#   1. Reconstruct every trade at 1-minute resolution so we have the full
#      indicator path for the duration of the hold — not just entry/exit prices.
#   2. Sweep exit parameter grids for both paths.
#   3. Compare fixed-hold benchmark vs best of each path on a uniform metric set.
# ==========================================
from src.indicators import fast_rolling_spearman

ENTRY_WINDOW  = 55        # bars used for entry signal (plateau result)
ENTRY_HR      = 0.80
ENTRY_SP      = 0.70
MAX_HOLD      = 60        # hard cap — never hold longer than this (minutes)
BENCHMARK_H   = 35        # fixed-hold benchmark

NY_START = "09:30"
NY_END   = "16:00"

print("=" * 80)
print("EXIT CRITERION RESEARCH")
print(f"  Entry : W={ENTRY_WINDOW}b | HR≥{ENTRY_HR} | SP≥{ENTRY_SP} | NY Session")
print(f"  Benchmark : fixed hold = {BENCHMARK_H} min")
print(f"  Hard cap  : {MAX_HOLD} min (applies to all dynamic exits)")
print("=" * 80)


# ──────────────────────────────────────────────────────────────────
# STEP 1 — Build the base indicator series for W=55
#           and collect every trade as a minute-by-minute path.
# ──────────────────────────────────────────────────────────────────
print("\n  [1/5] Building W=55 indicator series and trade paths …")

_base = df_ny[["btc_ret", "nq_ret", "nq_close", "btc_close"]].copy()

_base["sp_55"] = fast_rolling_spearman(
    _base["btc_ret"].to_numpy(),
    _base["nq_ret"].to_numpy(),
    ENTRY_WINDOW,
)
_dir_match = (np.sign(_base["btc_ret"]) == np.sign(_base["nq_ret"])).astype(float)
_base["hr_55"] = _dir_match.rolling(ENTRY_WINDOW).mean()
_base = _base.dropna(subset=["sp_55", "hr_55"])

# Session + threshold filter → event starts
_time_ok  = (
    (_base.index.time >= pd.to_datetime(NY_START).time()) &
    (_base.index.time <= pd.to_datetime(NY_END).time())
)
_valid    = _time_ok & (_base["hr_55"] >= ENTRY_HR) & (_base["sp_55"] >= ENTRY_SP)
_ev_start = _valid & (~_valid.shift(1, fill_value=False))
_entry_ts = _base.index[_ev_start]

print(f"  Entry signals : {len(_entry_ts)}")

# For each entry, store the full minute-by-minute path up to MAX_HOLD
_iloc_map = {ts: i for i, ts in enumerate(_base.index)}

class TradePath:
    """Minute-by-minute snapshot of price and indicators during a hold."""
    __slots__ = ("ts", "direction", "nq_entry", "nq_prices",
                 "hr_path", "sp_path", "ret_path_nq", "ret_path_btc")

trade_paths: list[TradePath] = []

for ts in _entry_ts:
    pos = _iloc_map[ts]
    # direction from entry bar momentum
    direction = np.sign(_base["nq_ret"].iloc[pos]) or 1.0

    end_pos = min(pos + MAX_HOLD + 1, len(_base))
    segment = _base.iloc[pos:end_pos]

    if len(segment) < 2:
        continue

    tp            = TradePath()
    tp.ts         = ts
    tp.direction  = direction
    tp.nq_entry   = segment["nq_close"].iloc[0]
    tp.nq_prices  = segment["nq_close"].to_numpy()
    tp.hr_path    = segment["hr_55"].to_numpy()
    tp.sp_path    = segment["sp_55"].to_numpy()

    # short-window (5-bar) directional agreement — used for divergence exit
    tp.ret_path_nq  = segment["nq_ret"].to_numpy()
    tp.ret_path_btc = segment["btc_ret"].to_numpy()

    trade_paths.append(tp)

print(f"  Trade paths built : {len(trade_paths)}")


# ──────────────────────────────────────────────────────────────────
# SHARED HELPERS
# ──────────────────────────────────────────────────────────────────
def _bps_at(tp: TradePath, minute: int) -> float:
    """Forward bps for trade tp exited at `minute` bars after entry."""
    idx = min(minute, len(tp.nq_prices) - 1)
    return (tp.nq_prices[idx] / tp.nq_entry - 1.0) * tp.direction * 10_000


def _equity_stats_from_bps(bps_list: list[float]) -> dict:
    bps = np.array(bps_list)
    if len(bps) == 0:
        return {}
    cum         = np.cumsum(bps)
    running_max = np.maximum.accumulate(cum)
    dd          = cum - running_max
    win_rate    = (bps > 0).mean() * 100
    avg_win     = bps[bps > 0].mean() if (bps > 0).any() else np.nan
    avg_loss    = bps[bps < 0].mean() if (bps < 0).any() else np.nan
    pf          = (bps[bps > 0].sum() / abs(bps[bps < 0].sum())
                   if (bps < 0).any() else np.inf)
    sharpe      = bps.mean() / bps.std() * np.sqrt(len(bps)) if bps.std() > 0 else np.nan
    sortino     = (bps.mean() / bps[bps < 0].std() * np.sqrt(len(bps))
                   if (bps < 0).any() else np.nan)
    max_dd      = float(dd.min())
    calmar      = (float(cum[-1]) / abs(max_dd)) if max_dd < 0 else np.inf
    in_dd       = (dd < 0).astype(int)
    # Longest consecutive run of bars spent underwater
    count, max_dd_dur = 0, 0
    for v in in_dd:
        count = count + 1 if v else 0
        if count > max_dd_dur:
            max_dd_dur = count
    dd_dur = max_dd_dur
    return dict(
        n_trades    = len(bps),
        win_rate_pct= round(float(win_rate), 2),
        avg_bps     = round(float(bps.mean()), 2),
        median_bps  = round(float(np.median(bps)), 2),
        avg_win_bps = round(float(avg_win), 2) if not np.isnan(avg_win) else np.nan,
        avg_loss_bps= round(float(avg_loss), 2) if not np.isnan(avg_loss) else np.nan,
        profit_factor= round(float(pf), 3),
        sharpe      = round(float(sharpe), 3),
        sortino     = round(float(sortino), 3),
        total_bps   = round(float(cum[-1]), 1),
        max_dd_bps  = round(max_dd, 1),
        calmar      = round(float(calmar), 3),
        cum_curve   = cum.tolist(),
    )


# Benchmark: fixed hold
_bench_bps = [_bps_at(tp, BENCHMARK_H) for tp in trade_paths]
bench_stats = _equity_stats_from_bps(_bench_bps)
bench_stats["label"] = f"Fixed Hold {BENCHMARK_H}m (benchmark)"

print(f"\n  Benchmark ({BENCHMARK_H}m fixed hold) :")
print(f"    n={bench_stats['n_trades']}  win={bench_stats['win_rate_pct']:.1f}%  "
      f"avg={bench_stats['avg_bps']:.2f} bps  sharpe={bench_stats['sharpe']:.3f}  "
      f"total={bench_stats['total_bps']:.0f} bps  maxDD={bench_stats['max_dd_bps']:.0f} bps")


# ──────────────────────────────────────────────────────────────────
# PATH A — CORRELATION DETERIORATION EXIT
# At each minute after entry, check if the rolling HR or SP has
# dropped below a configurable threshold.  Exit immediately.
# If neither threshold is breached within MAX_HOLD, exit at MAX_HOLD.
# ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("[PATH A]  CORRELATION DETERIORATION EXIT")
print("  Exit rule : exit at the first minute where EITHER")
print("              hr_55 < hr_exit_threshold  OR  sp_55 < sp_exit_threshold.")
print("  Fallback  : exit at MAX_HOLD if never breached.")
print("=" * 80)

# Sweep grid — exit thresholds can be at or below the entry thresholds
_hr_exit_vals = np.round(np.arange(0.50, 0.85, 0.05), 2)
_sp_exit_vals = np.round(np.arange(0.30, 0.75, 0.05), 2)

path_a_results = []
for hr_exit in _hr_exit_vals:
    for sp_exit in _sp_exit_vals:
        bps_list, hold_mins = [], []
        for tp in trade_paths:
            exit_min = MAX_HOLD
            for m in range(1, len(tp.hr_path)):
                if tp.hr_path[m] < hr_exit or tp.sp_path[m] < sp_exit:
                    exit_min = m
                    break
            bps_list.append(_bps_at(tp, exit_min))
            hold_mins.append(exit_min)
        stats = _equity_stats_from_bps(bps_list)
        stats["hr_exit"]       = hr_exit
        stats["sp_exit"]       = sp_exit
        stats["avg_hold_min"]  = round(np.mean(hold_mins), 1)
        stats["pct_early_exit"]= round(100 * sum(h < MAX_HOLD for h in hold_mins) / len(hold_mins), 1)
        path_a_results.append(stats)

path_a_df = pd.DataFrame(path_a_results)

# Heatmaps: sharpe, total_bps, avg_hold_min
def _heat_pivot(df, val_col, idx_col="hr_exit", col_col="sp_exit"):
    return df.pivot(index=idx_col, columns=col_col, values=val_col)

_a_metrics = {
    "Sharpe":          "sharpe",
    "Total BPS":       "total_bps",
    "Win Rate %":      "win_rate_pct",
    "Avg Hold (min)":  "avg_hold_min",
    "% Early Exit":    "pct_early_exit",
    "Max DD (bps)":    "max_dd_bps",
}

fig_a_heat = make_subplots(
    rows=2, cols=3,
    subplot_titles=list(_a_metrics.keys()),
    vertical_spacing=0.14, horizontal_spacing=0.10,
)
for idx, (title, col) in enumerate(_a_metrics.items()):
    r, c = divmod(idx, 3)
    pivot = _heat_pivot(path_a_df, col)
    z     = pivot.values.tolist()
    x     = [f"SP<{v:.2f}" for v in pivot.columns]
    y     = [f"HR<{v:.2f}" for v in pivot.index]
    text  = [[f"{v:.1f}" if v == v else "—" for v in row] for row in pivot.values]
    cmap  = "RdYlGn_r" if col == "max_dd_bps" else "RdYlGn"
    fig_a_heat.add_trace(
        go.Heatmap(z=z, x=x, y=y, text=text, texttemplate="%{text}",
                   colorscale=cmap, showscale=False,
                   hovertemplate=f"{title}<br>%{{y}}  %{{x}}<br>%{{z:.2f}}<extra></extra>"),
        row=r+1, col=c+1,
    )
# Benchmark reference lines (drawn as annotations)
fig_a_heat.update_layout(
    title=dict(
        text=(
            "Path A — Correlation Exit Parameter Sweep<br>"
            f"<sup>Entry HR≥{ENTRY_HR} SP≥{ENTRY_SP} W={ENTRY_WINDOW}  |  "
            f"Benchmark: fixed {BENCHMARK_H}m hold (Sharpe {bench_stats['sharpe']:.2f}, "
            f"Total {bench_stats['total_bps']:.0f} bps)</sup>"
        ),
        font=dict(size=13),
    ),
    height=680, template="plotly_dark",
)
fig_a_heat.show()

# Best Path A config by Sharpe
best_a = path_a_df.sort_values("sharpe", ascending=False).iloc[0]
print(f"\n  Best Path A config (by Sharpe):")
print(f"    hr_exit={best_a['hr_exit']:.2f}  sp_exit={best_a['sp_exit']:.2f}")
print(f"    n={best_a['n_trades']}  win={best_a['win_rate_pct']:.1f}%  "
      f"avg={best_a['avg_bps']:.2f} bps  sharpe={best_a['sharpe']:.3f}  "
      f"total={best_a['total_bps']:.0f} bps  maxDD={best_a['max_dd_bps']:.0f} bps")
print(f"    Avg hold = {best_a['avg_hold_min']:.1f} min  |  "
      f"Early exits = {best_a['pct_early_exit']:.1f}%")

# Collect best Path A bps for comparison chart
_best_a_bps = []
for tp in trade_paths:
    exit_min = MAX_HOLD
    for m in range(1, len(tp.hr_path)):
        if tp.hr_path[m] < best_a["hr_exit"] or tp.sp_path[m] < best_a["sp_exit"]:
            exit_min = m
            break
    _best_a_bps.append(_bps_at(tp, exit_min))
best_a_stats = _equity_stats_from_bps(_best_a_bps)
best_a_stats["label"] = f"Path A: corr exit (HR<{best_a['hr_exit']:.2f} OR SP<{best_a['sp_exit']:.2f})"


# ──────────────────────────────────────────────────────────────────
# PATH B — DIVERGENCE / REVERSAL EXIT
# Two sub-mechanisms tested:
#   B1. Short-window Spearman flip: compute a fast (N-bar) Spearman
#       over the LIVE bars during the hold.  Exit when it goes negative.
#   B2. Momentum reversal: exit when NQ has accumulated negative
#       returns over the last K bars (against position direction).
# ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("[PATH B]  DIVERGENCE / REVERSAL EXIT")
print("  B1. Fast Spearman flip : compute a short rolling Spearman (N bars)")
print("      over bars DURING the hold.  Exit when it turns negative.")
print("  B2. Momentum reversal  : exit when cumulative NQ return over")
print("      the last K bars turns against the position.")
print("=" * 80)

# ── B1: fast Spearman flip ──────────────────────────────────────
print("\n  [B1] Fast Spearman flip sweep …")
_sp_windows = [3, 5, 7, 10, 15]

path_b1_results = []
for sp_win in _sp_windows:
    bps_list, hold_mins = [], []
    for tp in trade_paths:
        exit_min = MAX_HOLD
        # need at least sp_win bars to compute fast Spearman
        for m in range(sp_win, min(len(tp.ret_path_nq), MAX_HOLD + 1)):
            x_slice = tp.ret_path_btc[m - sp_win: m]
            y_slice = tp.ret_path_nq[m - sp_win: m]
            if np.std(x_slice) < 1e-10 or np.std(y_slice) < 1e-10:
                continue
            # rank-based Spearman (inline, no statsmodels overhead)
            rx = np.argsort(np.argsort(x_slice))
            ry = np.argsort(np.argsort(y_slice))
            rm = (sp_win - 1) / 2.0
            rv = (sp_win ** 2 - 1) / 12.0
            sp = np.mean((rx - rm) * (ry - rm)) / rv
            if sp < 0:
                exit_min = m
                break
        bps_list.append(_bps_at(tp, exit_min))
        hold_mins.append(exit_min)
    stats = _equity_stats_from_bps(bps_list)
    stats["sp_window"]     = sp_win
    stats["avg_hold_min"]  = round(np.mean(hold_mins), 1)
    stats["pct_early_exit"]= round(100 * sum(h < MAX_HOLD for h in hold_mins) / len(hold_mins), 1)
    path_b1_results.append(stats)
    print(f"    sp_win={sp_win:2d} → sharpe={stats['sharpe']:.3f}  "
          f"total={stats['total_bps']:.0f} bps  avg_hold={stats['avg_hold_min']:.1f}m  "
          f"early={stats['pct_early_exit']:.1f}%")

# ── B2: momentum reversal ───────────────────────────────────────
print("\n  [B2] Momentum reversal sweep …")
_mom_windows   = [3, 5, 7, 10, 15]
_mom_thresholds = [-2, -5, -8, -12, -20]   # in bps — how far against before exit

path_b2_results = []
for k in _mom_windows:
    for thr_bps in _mom_thresholds:
        bps_list, hold_mins = [], []
        for tp in trade_paths:
            exit_min = MAX_HOLD
            for m in range(k, min(len(tp.nq_prices), MAX_HOLD + 1)):
                roll_ret = (tp.nq_prices[m] / tp.nq_prices[m - k] - 1.0) * tp.direction * 10_000
                if roll_ret < thr_bps:
                    exit_min = m
                    break
            bps_list.append(_bps_at(tp, exit_min))
            hold_mins.append(exit_min)
        stats = _equity_stats_from_bps(bps_list)
        stats["mom_window"]    = k
        stats["mom_thr_bps"]   = thr_bps
        stats["avg_hold_min"]  = round(np.mean(hold_mins), 1)
        stats["pct_early_exit"]= round(100 * sum(h < MAX_HOLD for h in hold_mins) / len(hold_mins), 1)
        path_b2_results.append(stats)

path_b2_df = pd.DataFrame(path_b2_results)

# B2 heatmap
fig_b2 = make_subplots(rows=1, cols=3,
                        subplot_titles=["Sharpe", "Total BPS", "Avg Hold (min)"],
                        horizontal_spacing=0.10)
for ci, col in enumerate(["sharpe", "total_bps", "avg_hold_min"], start=1):
    piv = path_b2_df.pivot(index="mom_window", columns="mom_thr_bps", values=col)
    z   = piv.values.tolist()
    x   = [f"≤{v}bps" for v in piv.columns]
    y   = [f"K={int(v)}b" for v in piv.index]
    txt = [[f"{v:.1f}" for v in row] for row in piv.values]
    fig_b2.add_trace(
        go.Heatmap(z=z, x=x, y=y, text=txt, texttemplate="%{text}",
                   colorscale="RdYlGn", showscale=False,
                   hovertemplate=f"{col}<br>%{{y}} %{{x}}: %{{z:.2f}}<extra></extra>"),
        row=1, col=ci,
    )
fig_b2.update_layout(
    title=dict(
        text=(
            "Path B2 — Momentum Reversal Exit Sweep<br>"
            "<sup>Rows = lookback window (bars) · Columns = drawdown threshold (bps against position)</sup>"
        ),
        font=dict(size=13),
    ),
    height=420, template="plotly_dark",
)
fig_b2.show()

best_b1 = pd.DataFrame(path_b1_results).sort_values("sharpe", ascending=False).iloc[0]
best_b2 = path_b2_df.sort_values("sharpe", ascending=False).iloc[0]

print(f"\n  Best B1 (fast Spearman flip): sp_win={int(best_b1['sp_window'])}b  "
      f"sharpe={best_b1['sharpe']:.3f}  total={best_b1['total_bps']:.0f} bps  "
      f"avg_hold={best_b1['avg_hold_min']:.1f}m  early={best_b1['pct_early_exit']:.1f}%")
print(f"  Best B2 (momentum reversal): K={int(best_b2['mom_window'])}b  "
      f"thr={best_b2['mom_thr_bps']:.0f}bps  sharpe={best_b2['sharpe']:.3f}  "
      f"total={best_b2['total_bps']:.0f} bps  avg_hold={best_b2['avg_hold_min']:.1f}m")

# Collect best B bps list (whichever is better)
best_b  = best_b1 if best_b1["sharpe"] >= best_b2["sharpe"] else best_b2
_best_b_bps = []
if best_b1["sharpe"] >= best_b2["sharpe"]:
    sp_win = int(best_b1["sp_window"])
    for tp in trade_paths:
        exit_min = MAX_HOLD
        for m in range(sp_win, min(len(tp.ret_path_nq), MAX_HOLD + 1)):
            x_slice = tp.ret_path_btc[m - sp_win: m]
            y_slice = tp.ret_path_nq[m - sp_win: m]
            if np.std(x_slice) < 1e-10 or np.std(y_slice) < 1e-10:
                continue
            rx  = np.argsort(np.argsort(x_slice))
            ry  = np.argsort(np.argsort(y_slice))
            rm  = (sp_win - 1) / 2.0
            rv  = (sp_win ** 2 - 1) / 12.0
            sp  = np.mean((rx - rm) * (ry - rm)) / rv
            if sp < 0:
                exit_min = m
                break
        _best_b_bps.append(_bps_at(tp, exit_min))
    best_b_label = f"Path B1: Spearman flip (win={sp_win}b)"
else:
    k_b    = int(best_b2["mom_window"])
    thr_b  = float(best_b2["mom_thr_bps"])
    for tp in trade_paths:
        exit_min = MAX_HOLD
        for m in range(k_b, min(len(tp.nq_prices), MAX_HOLD + 1)):
            roll_ret = (tp.nq_prices[m] / tp.nq_prices[m - k_b] - 1.0) * tp.direction * 10_000
            if roll_ret < thr_b:
                exit_min = m
                break
        _best_b_bps.append(_bps_at(tp, exit_min))
    best_b_label = f"Path B2: momentum reversal (K={k_b}b, thr={thr_b:.0f}bps)"

best_b_stats = _equity_stats_from_bps(_best_b_bps)
best_b_stats["label"] = best_b_label


# ──────────────────────────────────────────────────────────────────
# STEP 5 — COMPARISON : BENCHMARK vs PATH A vs PATH B
# ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("COMPARISON  —  Fixed Hold vs Correlation Exit vs Divergence Exit")
print("=" * 80)

_compare_configs = [bench_stats, best_a_stats, best_b_stats]
_cmp_cols = [
    "label", "n_trades", "win_rate_pct", "avg_bps", "median_bps",
    "avg_win_bps", "avg_loss_bps", "profit_factor",
    "sharpe", "sortino", "total_bps", "max_dd_bps", "calmar",
]
cmp_df = pd.DataFrame([
    {k: v for k, v in s.items() if k in _cmp_cols}
    for s in _compare_configs
]).set_index("label")

print(cmp_df.round(2).to_string())

# ── Overlaid equity curves ─────────────────────────────────────
fig_cmp = go.Figure()
_palette_cmp = {"bench": "#636EFA", "a": "#00CC96", "b": "#FFA15A"}

for stats, color, dash in [
    (bench_stats,  _palette_cmp["bench"], "dot"),
    (best_a_stats, _palette_cmp["a"],     "solid"),
    (best_b_stats, _palette_cmp["b"],     "solid"),
]:
    fig_cmp.add_trace(go.Scatter(
        x=list(range(len(stats["cum_curve"]))),
        y=stats["cum_curve"],
        mode="lines",
        name=stats["label"],
        line=dict(color=color, width=2.5, dash=dash),
        hovertemplate=stats["label"] + "<br>Trade #%{x}<br>Cum: %{y:.0f} bps<extra></extra>",
    ))

fig_cmp.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.25)")
fig_cmp.update_layout(
    title=dict(
        text=(
            "Equity Curves — Fixed Hold vs Correlation Exit vs Divergence Exit<br>"
            f"<sup>Entry: W={ENTRY_WINDOW}b | HR≥{ENTRY_HR} | SP≥{ENTRY_SP} | NY Session | "
            f"Hard cap {MAX_HOLD}m</sup>"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Trade Number",
    yaxis_title="Cumulative PnL (bps)",
    height=480, template="plotly_dark",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_cmp.show()

# ── Hold-time distributions ────────────────────────────────────
# Reconstruct hold times for Path A and Path B for the histogram
_a_holds, _b_holds = [], []
for tp in trade_paths:
    m = MAX_HOLD
    for i in range(1, len(tp.hr_path)):
        if tp.hr_path[i] < best_a["hr_exit"] or tp.sp_path[i] < best_a["sp_exit"]:
            m = i; break
    _a_holds.append(m)

for tp in trade_paths:
    m = MAX_HOLD
    if best_b_label.startswith("Path B1"):
        sp_win = int(best_b1["sp_window"])
        for i in range(sp_win, min(len(tp.ret_path_nq), MAX_HOLD + 1)):
            x_slice = tp.ret_path_btc[i - sp_win: i]
            y_slice = tp.ret_path_nq[i - sp_win: i]
            if np.std(x_slice) < 1e-10 or np.std(y_slice) < 1e-10:
                continue
            rx = np.argsort(np.argsort(x_slice)); ry = np.argsort(np.argsort(y_slice))
            rm = (sp_win - 1) / 2.0; rv = (sp_win ** 2 - 1) / 12.0
            if np.mean((rx - rm) * (ry - rm)) / rv < 0:
                m = i; break
    else:
        k_b   = int(best_b2["mom_window"]); thr_b = float(best_b2["mom_thr_bps"])
        for i in range(k_b, min(len(tp.nq_prices), MAX_HOLD + 1)):
            if (tp.nq_prices[i] / tp.nq_prices[i - k_b] - 1.0) * tp.direction * 10_000 < thr_b:
                m = i; break
    _b_holds.append(m)

fig_holds = go.Figure()
for holds, color, name in [
    ([BENCHMARK_H] * len(trade_paths), _palette_cmp["bench"], f"Fixed {BENCHMARK_H}m"),
    (_a_holds, _palette_cmp["a"], "Path A (corr exit)"),
    (_b_holds, _palette_cmp["b"], "Path B (div exit)"),
]:
    fig_holds.add_trace(go.Histogram(
        x=holds, nbinsx=MAX_HOLD,
        name=name,
        marker_color=color,
        opacity=0.65,
    ))
fig_holds.update_layout(
    barmode="overlay",
    title="Hold-Time Distribution — Fixed vs Dynamic Exits",
    xaxis_title="Minutes held",
    yaxis_title="Trade count",
    height=380, template="plotly_dark",
)
fig_holds.show()


# ──────────────────────────────────────────────────────────────────
# STEP 6 — SAMPLE TRADE WALKTHROUGH
# Show 6 randomly sampled trades with their per-minute indicator
# path and the three exit points annotated.
# ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("SAMPLE TRADE WALKTHROUGH  (6 trades — indicators + exit signals)")
print("=" * 80)

rng     = np.random.default_rng(42)
sample  = rng.choice(len(trade_paths), size=min(6, len(trade_paths)), replace=False)
sp_win_b1 = int(best_b1["sp_window"]) if best_b_label.startswith("Path B1") else 5

for sample_idx in sample:
    tp   = trade_paths[sample_idx]
    n    = min(len(tp.nq_prices), MAX_HOLD + 1)
    mins = list(range(n))

    # Cum bps path from entry
    cum_bps_path = [(tp.nq_prices[m] / tp.nq_entry - 1.0) * tp.direction * 10_000 for m in mins]

    # Reconstruct Path A exit
    exit_a = MAX_HOLD
    for m in range(1, len(tp.hr_path)):
        if tp.hr_path[m] < best_a["hr_exit"] or tp.sp_path[m] < best_a["sp_exit"]:
            exit_a = m; break

    # Reconstruct Path B exit
    exit_b = MAX_HOLD
    if best_b_label.startswith("Path B1"):
        for m in range(sp_win_b1, min(n, MAX_HOLD + 1)):
            x_s = tp.ret_path_btc[m - sp_win_b1: m]; y_s = tp.ret_path_nq[m - sp_win_b1: m]
            if np.std(x_s) < 1e-10 or np.std(y_s) < 1e-10: continue
            rx = np.argsort(np.argsort(x_s)); ry = np.argsort(np.argsort(y_s))
            rm = (sp_win_b1 - 1) / 2.0; rv = (sp_win_b1 ** 2 - 1) / 12.0
            if np.mean((rx - rm) * (ry - rm)) / rv < 0: exit_b = m; break
    else:
        k_b = int(best_b2["mom_window"]); thr_b = float(best_b2["mom_thr_bps"])
        for m in range(k_b, min(n, MAX_HOLD + 1)):
            if (tp.nq_prices[m] / tp.nq_prices[m - k_b] - 1.0) * tp.direction * 10_000 < thr_b:
                exit_b = m; break

    fig_samp = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f"NQ PnL path (bps)  |  Entry: {tp.ts.strftime('%Y-%m-%d %H:%M')}  "
            f"|  Direction: {'LONG' if tp.direction > 0 else 'SHORT'}",
            "Rolling HR & SP during hold",
        ],
        vertical_spacing=0.18,
    )

    # PnL path
    fig_samp.add_trace(
        go.Scatter(x=mins, y=cum_bps_path, mode="lines",
                   line=dict(color="#636EFA", width=2), name="NQ cum bps"),
        row=1, col=1,
    )
    fig_samp.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.3)", row=1, col=1)

    # Annotate the three exits
    for exit_min, color, tag in [
        (BENCHMARK_H, _palette_cmp["bench"], f"Fixed {BENCHMARK_H}m"),
        (exit_a,       _palette_cmp["a"],      f"Path A ({exit_a}m)"),
        (exit_b,       _palette_cmp["b"],      f"Path B ({exit_b}m)"),
    ]:
        em = min(exit_min, n - 1)
        fig_samp.add_vline(x=em, line_color=color, line_dash="dot", row=1, col=1)
        fig_samp.add_annotation(
            x=em, y=cum_bps_path[em],
            text=f"{tag}<br>{cum_bps_path[em]:.0f}bps",
            showarrow=True, arrowhead=2,
            font=dict(color=color, size=9),
            bgcolor="rgba(0,0,0,0.5)",
            row=1, col=1,
        )

    # HR and SP during hold
    fig_samp.add_trace(
        go.Scatter(x=mins[:len(tp.hr_path)], y=tp.hr_path[:n].tolist(),
                   mode="lines", name="HR", line=dict(color="#FFA15A", width=1.5)),
        row=2, col=1,
    )
    fig_samp.add_trace(
        go.Scatter(x=mins[:len(tp.sp_path)], y=tp.sp_path[:n].tolist(),
                   mode="lines", name="SP", line=dict(color="#00CC96", width=1.5)),
        row=2, col=1,
    )
    fig_samp.add_hline(y=ENTRY_HR, line_dash="dash", line_color=_palette_cmp["a"],
                       annotation_text=f"HR exit {best_a['hr_exit']:.2f}",
                       annotation_position="bottom right", row=2, col=1)
    fig_samp.add_hline(y=ENTRY_SP, line_dash="dash", line_color="#AB63FA",
                       annotation_text=f"SP exit {best_a['sp_exit']:.2f}",
                       annotation_position="top right", row=2, col=1)

    fig_samp.update_layout(
        height=520, template="plotly_dark",
        title=dict(
            text=f"Trade Walkthrough — Sample #{sample_idx}  |  "
                 f"W={ENTRY_WINDOW}b HR≥{ENTRY_HR} SP≥{ENTRY_SP} NY",
            font=dict(size=12),
        ),
        showlegend=True,
    )
    fig_samp.update_xaxes(title_text="Minutes after entry")
    fig_samp.update_yaxes(title_text="Cum PnL (bps)", row=1, col=1)
    fig_samp.update_yaxes(title_text="Indicator value", row=2, col=1)
    fig_samp.show()


# ── Final recommendation ──────────────────────────────────────
print("\n" + "=" * 80)
print("RECOMMENDATION")
print("=" * 80)
_winner = max(_compare_configs, key=lambda s: s["sharpe"])
print(f"\n  Highest Sharpe : {_winner['label']}")
print(f"    Sharpe={_winner['sharpe']:.3f}  Sortino={_winner['sortino']:.3f}  "
      f"Total={_winner['total_bps']:.0f}bps  MaxDD={_winner['max_dd_bps']:.0f}bps  "
      f"Calmar={_winner['calmar']:.3f}")
print(f"\n  Benchmark     : Sharpe={bench_stats['sharpe']:.3f}  "
      f"Total={bench_stats['total_bps']:.0f}bps  MaxDD={bench_stats['max_dd_bps']:.0f}bps")
print(f"  Path A winner : Sharpe={best_a_stats['sharpe']:.3f}  "
      f"Total={best_a_stats['total_bps']:.0f}bps  MaxDD={best_a_stats['max_dd_bps']:.0f}bps")
print(f"  Path B winner : Sharpe={best_b_stats['sharpe']:.3f}  "
      f"Total={best_b_stats['total_bps']:.0f}bps  MaxDD={best_b_stats['max_dd_bps']:.0f}bps")

In [ ]:
from IPython.display import HTML

# ==========================================
# FILTERED CREDIBLE CONFIGURATIONS ANALYSIS
# ==========================================
print("\n" + "=" * 80)
print("CREDIBLE CONFIGURATIONS ABOVE KEY THRESHOLDS (ROBUSTNESS SEARCH)")
print("=" * 80)

# --- Shared scoring helper ---
def _score(df: pd.DataFrame) -> pd.DataFrame:
    """Attach total_pnl_score = win_rate × events × avg_bps."""
    out = df.copy()
    out["total_pnl_score"] = out["oos_win_rate_pct"] * out["total_events"] * out["oos_avg_bps"]
    return out

_DISPLAY_COLS = [
    "window_bars", "hold_minutes", "hit_ratio_threshold", "spearman_threshold",
    "total_events", "oos_win_rate_pct", "oos_avg_bps", "total_pnl_score",
]
_SCROLLABLE = '<div style="max-height:400px;overflow-y:auto;border:1px solid #555;">{}</div>'

# [LIST 1] Win Rate > 50 % AND OOS avg bps > 0
_mask1 = (analysis_results["oos_win_rate_pct"] > 50) & (analysis_results["oos_avg_bps"] > 0)
list1_sorted = _score(analysis_results[_mask1]).sort_values("total_pnl_score", ascending=False)

print(f"\n[LIST 1] Win Rate > 50 % AND OOS Avg BPS > 0")
if list1_sorted.empty:
    print("  No configurations found.")
else:
    print(f"  {len(list1_sorted)} configurations | sorted by: win_rate × events × avg_bps")
    display(HTML(_SCROLLABLE.format(list1_sorted[_DISPLAY_COLS].round(2).to_html())))

# [LIST 2] Win Rate > 50 % AND OOS avg bps > 5  (survives slippage / costs)
_mask2 = (analysis_results["oos_win_rate_pct"] > 50) & (analysis_results["oos_avg_bps"] > 5)
list2_sorted = _score(analysis_results[_mask2]).sort_values("total_pnl_score", ascending=False)

print(f"\n[LIST 2] Win Rate > 50 % AND OOS Avg BPS > 5  (slippage-resilient universe)")
if list2_sorted.empty:
    print("  No configurations found.")
else:
    print(f"  {len(list2_sorted)} configurations | sorted by: win_rate × events × avg_bps")
    display(HTML(_SCROLLABLE.format(list2_sorted[_DISPLAY_COLS].round(2).to_html())))


# ==========================================
# THRESHOLD AGGREGATION TABLE
# Collapse window & hold — find which (hit_ratio, spearman) pairs
# are robust across ALL window/hold combinations in the >5 bps universe.
# ==========================================
if not list2_sorted.empty:
    print("\n" + "=" * 80)
    print("BEST THRESHOLD COMBINATIONS (WINDOW & HOLD COLLAPSED)")
    print("  What it shows : For every (hit_ratio, spearman) pair in the >5 bps list,")
    print("                  aggregate performance across ALL window/hold combinations.")
    print("  n_combos      : How many (window, hold) pairs passed the >5 bps filter.")
    print("  coverage_pct  : n_combos as a % of the full window×hold grid searched.")
    print("  Sorting       : Aggregate PnL score (sum of win_rate × events × avg_bps).")
    print("=" * 80)

    _total_grid = len(test_windows) * len(hold_horizons)

    threshold_table = (
        list2_sorted
        .groupby(["hit_ratio_threshold", "spearman_threshold"], sort=False)
        .agg(
            n_combos          = ("window_bars",      "count"),
            mean_oos_avg_bps  = ("oos_avg_bps",      "mean"),
            min_oos_avg_bps   = ("oos_avg_bps",      "min"),
            max_oos_avg_bps   = ("oos_avg_bps",      "max"),
            mean_win_rate_pct = ("oos_win_rate_pct", "mean"),
            total_events_sum  = ("total_events",     "sum"),
            agg_pnl_score     = ("total_pnl_score",  "sum"),
        )
        .reset_index()
    )
    threshold_table["coverage_pct"] = (threshold_table["n_combos"] / _total_grid * 100).round(1)
    threshold_table = threshold_table.sort_values("agg_pnl_score", ascending=False).reset_index(drop=True)
    threshold_table.index += 1  # 1-based rank

    _tbl_cols = [
        "hit_ratio_threshold", "spearman_threshold",
        "n_combos", "coverage_pct",
        "mean_oos_avg_bps", "min_oos_avg_bps", "max_oos_avg_bps",
        "mean_win_rate_pct", "total_events_sum", "agg_pnl_score",
    ]
    print(threshold_table[_tbl_cols].round(2).to_string())

    _best = threshold_table.iloc[0]
    print(
        f"\n  ★ Best threshold pair  →  "
        f"hit_ratio ≥ {_best['hit_ratio_threshold']:.2f}  |  "
        f"spearman ≥ {_best['spearman_threshold']:.2f}\n"
        f"     Passes filter in {int(_best['n_combos'])} / {_total_grid} "
        f"window×hold cells ({_best['coverage_pct']:.1f} % coverage)\n"
        f"     Mean OOS avg bps : {_best['mean_oos_avg_bps']:.2f}  "
        f"(range {_best['min_oos_avg_bps']:.2f} – {_best['max_oos_avg_bps']:.2f})\n"
        f"     Aggregate PnL score : {_best['agg_pnl_score']:,.0f}"
    )


# ==========================================
# MONTE CARLO–STYLE EQUITY CURVES
# All >5 bps configurations overlaid on a single chart.
# Curves are coloured by (hit_ratio, spearman) threshold pair so you
# can visually judge which threshold families dominate.
# ==========================================
def _calc_equity_curve(
    df: pd.DataFrame,
    hit_ratio_threshold: float,
    spearman_threshold: float,
    hold_minutes: int,
) -> np.ndarray | None:
    """
    Re-derive valid-signal edges from raw indicators (avoids dependence on
    the pre-baked event_start column which used the default thresholds).
    Returns a 1-D cumulative-bps array, or None if no trades fire.
    """
    valid = (df["hit_ratio"] >= hit_ratio_threshold) & (df["corr_spearman"] >= spearman_threshold)
    event_start = valid & (~valid.shift(1, fill_value=False))
    event_idx = df.index[event_start]

    if event_idx.empty:
        return None

    closes = df["nq_close"].to_numpy()
    rets   = df["nq_ret"].to_numpy()
    iloc_map = {ts: i for i, ts in enumerate(df.index)}

    bps_list = []
    for ts in event_idx:
        pos = iloc_map[ts]
        future = pos + hold_minutes
        if future >= len(closes):
            continue
        direction = np.sign(rets[pos]) or 1.0
        fwd_bps   = (closes[future] / closes[pos] - 1.0) * direction * 10_000
        bps_list.append(fwd_bps)

    return np.cumsum(bps_list) if bps_list else None


if not list2_sorted.empty:
    print("\n" + "=" * 80)
    print("MONTE CARLO–STYLE EQUITY CURVES  (all >5 bps configurations)")
    print(f"  Each line = one (window, hold, hit_ratio, spearman) configuration.")
    print(f"  Colour    = (hit_ratio, spearman) threshold pair — shared across all windows/holds.")
    print(f"  Bold line = cross-configuration median at each trade step.")
    print("=" * 80)

    # Assign a stable colour to every unique (HR, SP) threshold pair
    _threshold_pairs = (
        list2_sorted[["hit_ratio_threshold", "spearman_threshold"]]
        .drop_duplicates()
        .sort_values(["hit_ratio_threshold", "spearman_threshold"])
        .reset_index(drop=True)
    )
    _palette = px.colors.qualitative.Plotly + px.colors.qualitative.Safe
    _pair_color = {
        (row.hit_ratio_threshold, row.spearman_threshold): _palette[i % len(_palette)]
        for i, row in _threshold_pairs.iterrows()
    }

    # Build all equity curves
    _curves_meta: list[dict] = []
    for _, cfg in list2_sorted.iterrows():
        equity = _calc_equity_curve(
            signal_df,
            cfg["hit_ratio_threshold"],
            cfg["spearman_threshold"],
            int(cfg["hold_minutes"]),
        )
        if equity is not None:
            _curves_meta.append({
                "equity":  equity,
                "label":   (
                    f"HR≥{cfg['hit_ratio_threshold']:.2f}  SP≥{cfg['spearman_threshold']:.2f}  "
                    f"W={int(cfg['window_bars'])}  H={int(cfg['hold_minutes'])}"
                ),
                "color":   _pair_color[(cfg["hit_ratio_threshold"], cfg["spearman_threshold"])],
                "hr":      cfg["hit_ratio_threshold"],
                "sp":      cfg["spearman_threshold"],
                "final":   equity[-1],
            })

    print(f"  Curves plotted: {len(_curves_meta)} / {len(list2_sorted)}")

    # --- Individual curves (thin, semi-transparent) ---
    fig_mc = go.Figure()
    _seen_pairs: set = set()

    for meta in _curves_meta:
        pair = (meta["hr"], meta["sp"])
        show_legend = pair not in _seen_pairs
        _seen_pairs.add(pair)

        fig_mc.add_trace(go.Scatter(
            x=list(range(len(meta["equity"]))),
            y=meta["equity"].tolist(),
            mode="lines",
            name=f"HR≥{meta['hr']:.2f}  SP≥{meta['sp']:.2f}" if show_legend else None,
            legendgroup=f"{meta['hr']:.2f}_{meta['sp']:.2f}",
            showlegend=show_legend,
            line=dict(color=meta["color"], width=1),
            opacity=0.25,
            hovertemplate=meta["label"] + "<br>Trade #%{x}<br>Cum PnL: %{y:.0f} bps<extra></extra>",
        ))

    # --- Cross-configuration median line ---
    _max_len = max(len(m["equity"]) for m in _curves_meta)
    _padded  = [
        np.pad(m["equity"], (0, _max_len - len(m["equity"])), constant_values=np.nan)
        for m in _curves_meta
    ]
    _median_curve = np.nanmedian(_padded, axis=0)

    fig_mc.add_trace(go.Scatter(
        x=list(range(_max_len)),
        y=_median_curve.tolist(),
        mode="lines",
        name="Median (all configs)",
        line=dict(color="white", width=2.5, dash="dot"),
        opacity=1.0,
        hovertemplate="Median<br>Trade #%{x}<br>Cum PnL: %{y:.0f} bps<extra></extra>",
    ))

    # --- Zero reference ---
    fig_mc.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.3)")

    # Annotate final median value
    _final_median = float(_median_curve[~np.isnan(_median_curve)][-1])
    fig_mc.add_annotation(
        x=_max_len - 1, y=_final_median,
        text=f"Median final: {_final_median:.0f} bps",
        showarrow=True, arrowhead=2,
        font=dict(color="white", size=11),
        bgcolor="rgba(0,0,0,0.5)",
    )

    fig_mc.update_layout(
        title=dict(
            text=(
                f"Monte Carlo–Style Equity Curves — All {len(_curves_meta)} Configurations "
                f"(OOS avg bps > 5)<br>"
                f"<sup>Colour = threshold pair (HR, SP) · Window & hold collapsed into individual lines</sup>"
            ),
            font=dict(size=15),
        ),
        xaxis_title="Trade Number",
        yaxis_title="Cumulative PnL (bps)",
        height=650,
        template="plotly_dark",
        legend=dict(title="Threshold pair (HR, SP)", font=dict(size=10)),
        hovermode="closest",
    )
    fig_mc.show()

    # --- Final PnL distribution bar chart ---
    _finals_df = pd.DataFrame({
        "label":     [m["label"] for m in _curves_meta],
        "final_bps": [m["final"] for m in _curves_meta],
        "color":     [m["color"] for m in _curves_meta],
        "pair":      [f"HR≥{m['hr']:.2f} SP≥{m['sp']:.2f}" for m in _curves_meta],
    }).sort_values("final_bps", ascending=False)

    fig_dist = go.Figure(go.Bar(
        x=list(range(len(_finals_df))),
        y=_finals_df["final_bps"].tolist(),
        marker_color=_finals_df["color"].tolist(),
        opacity=0.85,
        hovertext=_finals_df["label"].tolist(),
        hovertemplate="%{hovertext}<br>Final PnL: %{y:.0f} bps<extra></extra>",
    ))
    fig_dist.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.4)")
    fig_dist.update_layout(
        title="Final Cumulative PnL Distribution — All >5 bps Configurations (ranked)",
        xaxis_title="Configuration Rank",
        yaxis_title="Final Cumulative PnL (bps)",
        height=400,
        template="plotly_dark",
        showlegend=False,
    )
    fig_dist.show()

    _n_positive = (_finals_df["final_bps"] > 0).sum()
    print(
        f"\n  Configurations with positive final equity : "
        f"{_n_positive} / {len(_finals_df)}  "
        f"({_n_positive / len(_finals_df) * 100:.1f} %)"
    )
    print(f"  Median final cumulative PnL : {_final_median:.0f} bps")
    print(f"  Best  final cumulative PnL  : {_finals_df['final_bps'].max():.0f} bps")
    print(f"  Worst final cumulative PnL  : {_finals_df['final_bps'].min():.0f} bps")

else:
    print("\nNo configurations in the >5 bps universe — nothing to plot.")
